# Week 6, Lecture 3: CNNs for Object Detection, Single-Stage Methods

**NPTEL: Deep Learning for Computer Vision**
Prof. Vineeth N Balasubramanian, Department of Computer Science and Engineering, IIT Hyderabad

Lecture 2 covered the **region-proposal (two-stage)** family: R-CNN, Fast R-CNN, Faster R-CNN. Those methods first *propose* a small set of candidate regions, then classify them. This lecture covers the other branch of the tree: **single-stage (dense prediction)** detectors, which skip the proposal step entirely and predict boxes directly on a dense grid.

## Topics

1. Warm-up: why smooth L1 is less sensitive to outliers than L2 (exercise from Lecture 2)
2. Recap: region proposal vs dense sampling, and the trade-off
3. **YOLO v1**: unified detection on an $S \times S$ grid
4. YOLO v1 bounding boxes and confidence, the $S \times S \times (B \cdot 5 + C)$ output tensor
5. YOLO v1 conditional class probabilities
6. YOLO v1 loss, term by term ($\lambda_{\text{coord}}$, $\lambda_{\text{noobj}}$, the $\sqrt{w}$ trick)
7. YOLO v1 limitations
8. **YOLO v2**: anchor boxes, the sigmoid/exp decode, k-means priors
9. **YOLO v3**: logistic objectness, multi-label classifiers, 3 scales, Darknet-53, SPP
10. YOLO anatomy: Backbone / Neck / Head
11. **YOLO v4**: CSPDarknet53, Mish, Bag-of-Specials, Bag-of-Freebies
12. YOLO versions timeline (v1 to v10)
13. **SSD**: multi-scale feature maps, exclusive predictors, default boxes
14. SSD loss: localization (smooth L1 on offsets) + confidence (softmax)
15. SSD tricks: hard negative mining, data augmentation
16. **FPN**: the top-down pathway
17. **RetinaNet**: why dense detection has a class imbalance problem
18. The problems caused by class imbalance
19. Why cross entropy is the wrong loss here
20. Balanced CE and **focal loss**
21. RetinaNet architecture = FPN + focal loss
22. Detectron / Detectron2

## What you will build

This notebook is built **bottom-up**: we peel the onion and assemble a working detector from parts, rather than calling a library.

- A **synthetic detection dataset** generated procedurally (coloured shapes with exact ground truth boxes). No downloads.
- The **grid target encoding**: assign each ground truth box to the cell containing its centre, encode $(x, y, w, h)$, and a matching decoder. We *assert* the encode/decode round trip is lossless.
- A **tiny conv backbone + detection head** producing a real $S \times S \times (B \cdot 5 + C)$ tensor.
- The **YOLO v1 loss, one term at a time**, each implemented and explained separately, then summed.
- **Training** on CPU in well under a minute, with real detections appearing on real shapes.
- **Decode, confidence threshold, and NMS** written from scratch (cross-checked against `torchvision`).
- The **YOLO v2 anchor decode** ($\sigma$ and $\exp$) applied to the same model, shown geometrically.
- A **Feature Pyramid Network from scratch** on a toy backbone, with real tensor shapes printed.
- **Focal loss from scratch**, and a 100k-anchor imbalance experiment showing exactly how much of the total loss easy negatives soak up under CE vs focal loss.

Everything runs on CPU in a few minutes. Interactive sliders appear throughout: **run every cell in order**.

## Setup

- PyTorch for all network code, numpy for data generation, matplotlib for plots, ipywidgets for sliders.
- We fix random seeds so every number quoted in the text is reproducible.
- `device` picks GPU if Colab gives you one, else CPU. The notebook is sized to run comfortably on **CPU**.
- The colour palette below is fixed and colourblind-safe; classes are always drawn in the same colour so the figures read consistently.

In [ ]:
%matplotlib inline
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)

# Fixed categorical palette (assigned in this order, never cycled).
C_BLUE, C_AQUA, C_YELLOW = '#2a78d6', '#1baf7a', '#eda100'
C_GREEN, C_VIOLET, C_RED = '#008300', '#4a3aa7', '#e34948'
PALETTE = [C_BLUE, C_AQUA, C_YELLOW, C_GREEN, C_VIOLET, C_RED]
INK, INK2, MUTED, GRIDC = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
# One-hue sequential ramp (light to dark) for magnitude heatmaps.
CMAP_BLUE = LinearSegmentedColormap.from_list('blue_seq', [
    '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#2a78d6', '#256abf', '#184f95', '#0d366b'])
# Ordinal ramp for ordered quantities (e.g. increasing gamma).
RAMP_ORDINAL = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'grid.color': GRIDC, 'grid.linewidth': 0.8,
    'axes.grid': True, 'axes.axisbelow': True,
    'font.size': 10, 'axes.titlesize': 11, 'legend.frameon': False,
    'lines.linewidth': 2.0, 'figure.dpi': 100,
})
print('setup done')

---
# 1. Warm-up: why is smooth L1 less sensitive to outliers than L2?

This was the exercise left at the end of Lecture 2. The answer, in one line:

> If the deviation of the predicted output from ground truth is very high, **squaring the difference explodes the gradient**. This happens with L2 loss for outliers, and is mitigated in the smooth L1 loss.

The two losses, for a residual $z = \hat{y} - y$:

$$
L_2(z) = z^2
\qquad\qquad
\text{smooth}_{L_1}(z) =
\begin{cases}
0.5\, z^2 / \beta & \text{if } |z| < \beta \\
|z| - 0.5\,\beta & \text{otherwise}
\end{cases}
$$

The point is not the loss value, it is the **gradient**:

$$
\frac{d L_2}{dz} = 2z \quad \text{(unbounded)}
\qquad\qquad
\frac{d\,\text{smooth}_{L_1}}{dz} =
\begin{cases}
z/\beta & \text{if } |z| < \beta \\
\text{sign}(z) & \text{otherwise}
\end{cases}
\quad \text{(bounded by 1)}
$$

- **L2**: gradient grows linearly with the residual, without limit. A single mislabelled or occluded box with $z = 50$ produces a gradient of 100, which can dominate the entire minibatch and destabilise training.
- **Smooth L1**: quadratic near zero (so it is smooth at the origin and gives fine-grained gradients for small errors, unlike plain L1), but **linear far from zero**, so the gradient saturates at magnitude 1. An outlier contributes the same gradient as a moderately wrong prediction: it can pull, but it cannot bully.
- This is exactly why Fast R-CNN and SSD regress box offsets with smooth L1 rather than L2.

Let us plot both, and their gradients.

In [ ]:
z = np.linspace(-4, 4, 801)
beta = 1.0

l2 = z**2
sl1 = np.where(np.abs(z) < beta, 0.5 * z**2 / beta, np.abs(z) - 0.5 * beta)
g_l2 = 2 * z
g_sl1 = np.where(np.abs(z) < beta, z / beta, np.sign(z))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

ax[0].plot(z, l2, color=C_RED, label='L2:  $z^2$')
ax[0].plot(z, sl1, color=C_BLUE, label=r'smooth L1 ($\beta$=1)')
ax[0].set_title('Loss value'); ax[0].set_xlabel('residual  z'); ax[0].set_ylabel('loss')
ax[0].set_ylim(0, 8); ax[0].legend()
ax[0].annotate('L2 explodes', xy=(2.6, 6.8), xytext=(0.4, 6.9), color=C_RED, fontsize=9,
               arrowprops=dict(arrowstyle='->', color=C_RED, lw=1.2))

ax[1].plot(z, g_l2, color=C_RED, label='dL2/dz = 2z')
ax[1].plot(z, g_sl1, color=C_BLUE, label='d(smooth L1)/dz')
ax[1].axhline(1, color=MUTED, ls=':', lw=1); ax[1].axhline(-1, color=MUTED, ls=':', lw=1)
ax[1].set_title('Gradient (this is the point)'); ax[1].set_xlabel('residual  z'); ax[1].set_ylabel('d loss / dz')
ax[1].set_ylim(-8, 8); ax[1].legend()
ax[1].annotate('bounded at $\\pm$1', xy=(3.0, 1.0), xytext=(1.2, 3.2), color=C_BLUE, fontsize=9,
               arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.2))

plt.suptitle('Smooth L1 vs L2: equal near the origin, totally different in the tails', y=1.02)
plt.tight_layout(); plt.show()

for zz in [0.5, 1.0, 5.0, 20.0]:
    print('residual %5.1f  ->  |dL2/dz| = %7.1f   |d smoothL1/dz| = %4.1f' % (zz, abs(2*zz), min(abs(zz), 1.0)))

The printout makes the asymmetry concrete: at a residual of 20 the L2 gradient is **40**, while smooth L1 still contributes exactly **1**. Move the slider below to see how $\beta$ sets the width of the quadratic region: as $\beta \to 0$ smooth L1 approaches plain L1 (kinked at the origin), and as $\beta$ grows it behaves like a scaled L2 over a wider range.

In [ ]:
def show_smooth_l1(beta=1.0):
    z = np.linspace(-4, 4, 801)
    sl1 = np.where(np.abs(z) < beta, 0.5 * z**2 / beta, np.abs(z) - 0.5 * beta)
    g = np.where(np.abs(z) < beta, z / beta, np.sign(z))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(z, z**2, color=C_RED, alpha=0.35, label='L2')
    ax[0].plot(z, sl1, color=C_BLUE, label='smooth L1')
    ax[0].axvspan(-beta, beta, color=C_BLUE, alpha=0.07)
    ax[0].set_ylim(0, 8); ax[0].set_title('loss  (shaded = quadratic region)'); ax[0].legend()
    ax[1].plot(z, g, color=C_BLUE, label='d(smooth L1)/dz')
    ax[1].axvspan(-beta, beta, color=C_BLUE, alpha=0.07)
    ax[1].set_ylim(-1.6, 1.6); ax[1].set_title('gradient, always in [-1, 1]'); ax[1].legend()
    for a in ax: a.set_xlabel('residual  z')
    plt.tight_layout(); plt.show()

interact(show_smooth_l1, beta=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                                  description='beta', continuous_update=False));

---
# 2. Recap: region proposal vs dense sampling

**Region proposal based (two-stage)**, from Lecture 2:
- In the first stage, potential object regions are proposed (Selective Search, or a Region Proposal Network).
- In the second stage, a classifier processes the candidate regions.
- More robust in performance, but **slower**.

**Dense sampling based (one-stage)**, this lecture:
- Integrates region proposal and detection into a single network acting on a **dense sampling of possible locations**.
- Simple and fast, but performance historically not as good as region-proposal methods.

| | Two-stage (Faster R-CNN) | Single-stage (YOLO, SSD) |
|---|---|---|
| Candidate regions | learned/searched proposals, ~1k to 2k | every cell of a fixed dense grid, ~10k to 100k |
| Passes over features | two (propose, then classify) | one |
| Foreground : background | roughly balanced *after* proposals filter | extremely skewed, most locations are background |
| Speed | slower | real time |
| Accuracy (2016 era) | higher | lower |

The last two rows are the whole story of this lecture:
- Dropping the proposal stage is what **buys the speed**.
- Dropping the proposal stage is also what **causes the class imbalance**, because nothing is filtering out the easy background any more.
- Sections 3 to 15 build the fast dense detectors. Sections 17 to 21 (RetinaNet, focal loss) go back and fix the accuracy gap that the imbalance caused. Keep this tension in mind: it is the reason the lecture ends where it does.

---
# 3. YOLO v1: unified detection

> Redmon et al, *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016

- A single-stage detector based on **OverFeat** (Lecture 1).
- **Speed with good performance** was the main aim. The whole pipeline is: resize image, run one convolutional network, non-max suppression. That is it.
- **Unified detection**: divide the image into an $S \times S$ grid.
  - The grid cell containing the **centre** of an object is *responsible* for detecting that object.
  - Each grid cell predicts $B$ bounding boxes and a confidence score for each.
  - In addition, each grid cell predicts $C$ conditional class probabilities, $\Pr(\text{Class}_i \mid \text{Object})$.
- Predictions for the whole image are therefore encoded as one tensor of shape $S \times S \times (B \cdot 5 + C)$.

## Our synthetic dataset

To keep this notebook self-contained (no downloads) we generate detection data procedurally: coloured shapes on a dark canvas, with **exact** ground truth boxes by construction.

- 3 classes: **rectangle**, **ellipse**, **triangle**. Each class always uses the same colour, and each shape has an independently sampled width and height, so the network must genuinely regress $w$ and $h$ rather than memorise one box size.
- 1 to 3 objects per $96 \times 96$ image, rejection-sampled so that they **do not overlap** and **no two centres fall in the same grid cell**.
- That last constraint is deliberate. YOLO v1 can only ever emit one class per cell, so two centres in one cell is a target the architecture *cannot* represent. We sidestep it by construction here, and in Section 7 we come back to it as a real limitation.
- This task is intentionally easy: it must converge in a few hundred CPU steps so you can watch detection actually work.

In [ ]:
IMG = 96          # image side in pixels
S = 6             # grid is S x S
CELL = IMG // S   # 16 pixels per cell
B = 2             # boxes predicted per cell
C = 3             # number of classes
CLASS_NAMES = ['rectangle', 'ellipse', 'triangle']
CLASS_HEX = [C_BLUE, C_AQUA, C_YELLOW]

def _hex2rgb(h):
    h = h.lstrip('#')
    return np.array([int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)], np.float32)
CLASS_RGB = [_hex2rgb(h) for h in CLASS_HEX]

_yy, _xx = np.mgrid[0:IMG, 0:IMG]
_yy = _yy + 0.5; _xx = _xx + 0.5   # pixel centres

def sample_boxes(rng, n_obj):
    "Rejection-sample non-overlapping boxes whose centres land in distinct cells."
    boxes = []
    for _ in range(200):
        if len(boxes) == n_obj:
            break
        w, h = rng.uniform(18, 34), rng.uniform(18, 34)
        cx = rng.uniform(w/2 + 2, IMG - w/2 - 2)
        cy = rng.uniform(h/2 + 2, IMG - h/2 - 2)
        ci, cj = int(cy // CELL), int(cx // CELL)
        ok = True
        for (bcx, bcy, bw, bh, _c) in boxes:
            if abs(cx - bcx) < (w + bw)/2 + 3 and abs(cy - bcy) < (h + bh)/2 + 3:
                ok = False; break                      # no pixel overlap
            if (int(bcy // CELL), int(bcx // CELL)) == (ci, cj):
                ok = False; break                      # one object per cell
        if ok:
            boxes.append((cx, cy, w, h, int(rng.integers(0, C))))
    return boxes

def render(boxes, rng):
    "Rasterise shapes. Box (cx, cy, w, h) is the exact bounding box of each shape."
    img = np.full((IMG, IMG, 3), 0.10, np.float32)
    img += 0.03 * rng.standard_normal((IMG, IMG, 1)).astype(np.float32)
    for (cx, cy, w, h, cls) in boxes:
        if cls == 0:                                   # rectangle
            m = (np.abs(_xx - cx) <= w/2) & (np.abs(_yy - cy) <= h/2)
        elif cls == 1:                                 # ellipse
            m = (((_xx - cx)/(w/2))**2 + ((_yy - cy)/(h/2))**2) <= 1.0
        else:                                          # triangle, apex up
            top, bot = cy - h/2, cy + h/2
            half = (w/2) * (_yy - top) / h
            m = (_yy >= top) & (_yy <= bot) & (np.abs(_xx - cx) <= half)
        img[m] = CLASS_RGB[cls] * rng.uniform(0.85, 1.0)
    return np.clip(img, 0, 1)

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, IMG, IMG, 3), np.float32)
    boxes_all = []
    for k in range(n):
        bs = sample_boxes(rng, int(rng.integers(1, 4)))
        X[k] = render(bs, rng)
        boxes_all.append(bs)
    return X, boxes_all

t0 = time.time()
X_train, BX_train = make_dataset(512, seed=10)
X_test,  BX_test  = make_dataset(64,  seed=11)
print('generated %d train + %d test images of %dx%d in %.2fs'
      % (len(X_train), len(X_test), IMG, IMG, time.time() - t0))
print('objects per image (train): mean %.2f, min %d, max %d'
      % (np.mean([len(b) for b in BX_train]), min(len(b) for b in BX_train), max(len(b) for b in BX_train)))

Let us look at the data with its ground truth boxes. Each box is labelled with its class name, so class identity is never carried by colour alone.

In [ ]:
def draw_boxes(ax, boxes, lw=1.6, ls='-', label_scores=None):
    '''boxes: list of (cx, cy, w, h, cls) or (cx, cy, w, h, score, cls).
    The format is detected from the tuple length. label_scores=None means "label if a score
    is present"; pass False to draw boxes with no text at all.'''
    for b in boxes:
        if len(b) == 6:
            cx, cy, w, h, score, cls = b
            txt = '%s %.2f' % (CLASS_NAMES[int(cls)][:4], score)
        else:
            cx, cy, w, h, cls = b
            txt = CLASS_NAMES[int(cls)][:4]
        cls = int(cls)
        ax.add_patch(patches.Rectangle((cx - w/2, cy - h/2), w, h, fill=False,
                                       edgecolor=CLASS_HEX[cls], lw=lw, ls=ls))
        if label_scores is not False:
            ax.text(cx - w/2, cy - h/2 - 1.5, txt, color=CLASS_HEX[cls], fontsize=7,
                    ha='left', va='bottom', weight='bold')

fig, axes = plt.subplots(1, 5, figsize=(14, 3.1))
for k, ax in enumerate(axes):
    ax.imshow(X_train[k]); draw_boxes(ax, BX_train[k])
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title('%d object(s)' % len(BX_train[k]), fontsize=9)
plt.suptitle('Synthetic detection data with exact ground truth boxes', y=1.04)
plt.tight_layout(); plt.show()

## The grid and the responsible cell

The single most important idea in YOLO v1:

- The image is divided into an $S \times S$ grid ($6 \times 6$ here, so each cell covers $16 \times 16$ pixels).
- **The cell that contains the centre of an object is responsible for detecting that object.**
- Every other cell is expected to report "no object here".

Below, the grid is overlaid on an image, the responsible cell for each object is filled in, and the object centre is marked. Note that a box is usually much **larger** than the cell responsible for it: the cell owns the *centre*, not the extent.

In [ ]:
def draw_grid(ax, S_=S, color=MUTED, lw=0.6, alpha=0.9):
    step = IMG / S_
    for i in range(1, S_):
        ax.axhline(i * step, color=color, lw=lw, alpha=alpha)
        ax.axvline(i * step, color=color, lw=lw, alpha=alpha)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.9))
for k, ax in enumerate(axes):
    ax.imshow(X_train[k]); draw_grid(ax)
    for (cx, cy, w, h, cls) in BX_train[k]:
        j, i = int(cx // CELL), int(cy // CELL)
        ax.add_patch(patches.Rectangle((j*CELL, i*CELL), CELL, CELL,
                                       facecolor=CLASS_HEX[cls], alpha=0.45, edgecolor='white', lw=1.2))
        ax.plot([cx], [cy], marker='o', ms=5, color='white', mec=INK, mew=1.0)
    draw_boxes(ax, BX_train[k])
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title('responsible cells filled', fontsize=9)
plt.suptitle('The cell containing the object centre (white dot) is responsible for it', y=1.03)
plt.tight_layout(); plt.show()

### Slider: how $S$ and $B$ set the size of the output tensor

The network's entire output is one tensor of shape $S \times S \times (B \cdot 5 + C)$. The $5$ is $(x, y, w, h, \text{confidence})$ per box. Note the asymmetry that defines YOLO v1: **$B$ multiplies the box terms but not the class terms**, because a cell predicts only one set of class probabilities no matter how many boxes it has.

Move the sliders to see the grid get finer and the tensor grow. The original paper used $S=7$, $B=2$, $C=20$ on PASCAL VOC, giving $7 \times 7 \times 30$.

In [ ]:
def show_grid_tensor(S_=6, B_=2, C_=3):
    depth = B_ * 5 + C_
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={'width_ratios': [1, 1.25]})
    ax[0].imshow(X_train[1]); draw_grid(ax[0], S_)
    step = IMG / S_
    for (cx, cy, w, h, cls) in BX_train[1]:
        j, i = int(cx // step), int(cy // step)
        ax[0].add_patch(patches.Rectangle((j*step, i*step), step, step,
                                          facecolor=CLASS_HEX[cls], alpha=0.45, edgecolor='white', lw=1.0))
    draw_boxes(ax[0], BX_train[1])
    ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].grid(False)
    ax[0].set_title('S = %d  ->  %d cells' % (S_, S_*S_), fontsize=10)

    ax[1].axis('off')
    ax[1].set_title('output tensor', fontsize=10)
    ax[1].text(0.0, 0.80, r'$S \times S \times (B \cdot 5 + C)$', fontsize=15, color=INK)
    ax[1].text(0.0, 0.58, '= %d x %d x (%d*5 + %d)' % (S_, S_, B_, C_), fontsize=13, color=INK2)
    ax[1].text(0.0, 0.38, '= %d x %d x %d' % (S_, S_, depth), fontsize=15, color=C_BLUE, weight='bold')
    ax[1].text(0.0, 0.18, '= %d numbers per image' % (S_*S_*depth), fontsize=12, color=INK2)
    ax[1].text(0.0, 0.02, 'max detectable objects = %d (one class per cell)' % (S_*S_),
               fontsize=9, color=MUTED)
    ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

interact(show_grid_tensor,
         S_=widgets.IntSlider(value=6, min=2, max=13, step=1, description='S', continuous_update=False),
         B_=widgets.IntSlider(value=2, min=1, max=5, step=1, description='B', continuous_update=False),
         C_=widgets.IntSlider(value=3, min=1, max=20, step=1, description='C', continuous_update=False));

---
# 4. YOLO v1: bounding boxes and confidence scores

**Bounding boxes.** Each bounding box gives 4 coordinates $x, y, w, h$:
- $(x, y)$ = coordinates representing the **centre of the box relative to the grid cell**. Both lie in $[0, 1]$: $(0,0)$ is the top-left of *that cell*, $(1,1)$ its bottom-right.
- $(w, h)$ = width and height of the object **relative to the whole image**. Also in $[0, 1]$.

The two are normalised against **different references**, which is easy to get wrong when implementing. It is deliberate: a centre is a local quantity that a cell can reason about, while an extent may be much larger than any one cell.

**Confidence score.** Reflects how confident the model is that the box contains an object, *and* how accurate it thinks the box is. Formally:

$$
\text{Confidence} = \Pr(\text{Object}) \times \text{IOU}^{\text{truth}}_{\text{pred}}
$$

- If no object exists in that cell, confidence should be **zero**.
- If an object does exist, confidence should equal the **IoU between the predicted box and the ground truth**. So the network is asked to predict its own accuracy, and a box that is right about *there being an object* but sloppy about *where* is correctly penalised.
- We honour this below: the regression target for confidence is the live IoU, recomputed every step. This matters, and many reimplementations quietly replace it with a constant 1.

## The output tensor layout

For our $B = 2$, $C = 3$ setting each cell owns $2 \cdot 5 + 3 = 13$ numbers, laid out as below.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.0))
labels = ([r'$x_1$', r'$y_1$', r'$w_1$', r'$h_1$', r'$C_1$'] +
          [r'$x_2$', r'$y_2$', r'$w_2$', r'$h_2$', r'$C_2$'] +
          [r'$p_{rect}$', r'$p_{ellip}$', r'$p_{tri}$'])
groups = [(0, 5, C_BLUE, 'box 1:  (x, y, w, h, confidence)'),
          (5, 10, C_AQUA, 'box 2:  (x, y, w, h, confidence)'),
          (10, 13, C_YELLOW, 'C class probabilities (shared by both boxes)')]
for a, b, col, name in groups:
    ax.add_patch(patches.Rectangle((a + 0.04, 0.0), (b - a) - 0.08, 1.0,
                                   facecolor=col, alpha=0.18, edgecolor=col, lw=1.5))
    ax.text((a + b) / 2, 1.22, name, ha='center', fontsize=9, color=col, weight='bold')
for i, l in enumerate(labels):
    ax.add_patch(patches.Rectangle((i + 0.10, 0.12), 0.80, 0.76, facecolor='white',
                                   edgecolor=MUTED, lw=0.8))
    ax.text(i + 0.5, 0.5, l, ha='center', va='center', fontsize=11, color=INK)
    ax.text(i + 0.5, -0.16, str(i), ha='center', va='center', fontsize=7.5, color=MUTED)
ax.set_xlim(-0.2, 13.2); ax.set_ylim(-0.45, 1.55); ax.axis('off')
ax.set_title('Channel layout for ONE grid cell:  B*5 + C = 2*5 + 3 = 13 numbers  '
             '(the full tensor is 6 x 6 x 13)', fontsize=10, pad=18)
plt.tight_layout(); plt.show()
print('Full output tensor per image: %d x %d x %d = %d numbers' % (S, S, B*5+C, S*S*(B*5+C)))

---
# 5. YOLO v1: conditional class probabilities

- Regardless of the number of boxes $B$, we only predict **one set of class probabilities per grid cell**. Those probabilities are *conditional on an object being present*: $\Pr(\text{Class}_i \mid \text{Object})$.
- This is why the tensor depth is $B \cdot 5 + C$ and not $B \cdot (5 + C)$. It is a modelling choice with a real consequence: the $B$ boxes in a cell are alternative *hypotheses about one object's extent*, not slots for $B$ different objects. A cell can never emit two different classes.
- At test time, the class-specific confidence score for each box is obtained by multiplying the two:

$$
\Pr(\text{Class}_i \mid \text{Object}) \times \Pr(\text{Object}) \times \text{IOU}^{\text{truth}}_{\text{pred}}
= \Pr(\text{Class}_i) \times \text{IOU}^{\text{truth}}_{\text{pred}}
$$

- The middle two factors are exactly the confidence the box already predicts, so in code this is simply `class_prob * box_confidence`.
- These scores encode **both** the probability of that class appearing in the box **and** how well the box fits the object. That single number is what we threshold and feed to NMS in Section 6.6.

## 5.1 Target encoding, and proving it is lossless

Before any network, we need the function that turns a list of ground truth boxes into the target tensor, and its inverse. Getting this right is the foundation: **if the encoding is wrong, no amount of training will help**, and the bug is nearly invisible because the loss still goes down.

For a ground truth box with centre $(c_x, c_y)$ and size $(w, h)$ in pixels:

$$
j = \left\lfloor c_x / \text{CELL} \right\rfloor, \quad
i = \left\lfloor c_y / \text{CELL} \right\rfloor
$$
$$
x = c_x / \text{CELL} - j, \qquad y = c_y / \text{CELL} - i \qquad \text{(offset within the cell, in [0,1))}
$$
$$
w_{\text{enc}} = w / \text{IMG}, \qquad h_{\text{enc}} = h / \text{IMG} \qquad \text{(fraction of the whole image)}
$$

and the decoder inverts it:

$$
c_x = (j + x) \cdot \text{CELL}, \quad c_y = (i + y) \cdot \text{CELL}, \quad w = w_{\text{enc}} \cdot \text{IMG}, \quad h = h_{\text{enc}} \cdot \text{IMG}
$$

We then **assert** that `decode(encode(boxes)) == boxes` over many random samples. This is the round trip that proves our target representation can express the ground truth exactly.

In [ ]:
def encode(boxes):
    "boxes -> target tensor (S, S, 5 + C).  Channels: x, y, w, h, objectness, one-hot class."
    t = np.zeros((S, S, 5 + C), np.float32)
    for (cx, cy, w, h, cls) in boxes:
        j, i = int(cx // CELL), int(cy // CELL)     # responsible cell
        t[i, j, 0] = cx / CELL - j                  # x: centre offset within cell
        t[i, j, 1] = cy / CELL - i                  # y: centre offset within cell
        t[i, j, 2] = w / IMG                        # w: relative to whole image
        t[i, j, 3] = h / IMG                        # h: relative to whole image
        t[i, j, 4] = 1.0                            # objectness
        t[i, j, 5 + cls] = 1.0                      # one-hot class
    return t

def decode(t, thresh=0.5):
    "target/prediction tensor (S, S, 5 + C) -> list of (cx, cy, w, h, cls) in pixels."
    out = []
    for i in range(S):
        for j in range(S):
            if t[i, j, 4] > thresh:
                x, y, w, h = t[i, j, :4]
                out.append(((j + x) * CELL, (i + y) * CELL, w * IMG, h * IMG,
                            int(np.argmax(t[i, j, 5:]))))
    return out

# ---- round trip assertion -------------------------------------------------
rng = np.random.default_rng(123)
n_checked = 0
for _ in range(300):
    bs = sample_boxes(rng, int(rng.integers(1, 4)))
    rec = decode(encode(bs))
    assert len(rec) == len(bs), 'lost a box: %d in, %d out' % (len(bs), len(rec))
    for gt in bs:
        match = [r for r in rec if np.allclose(gt[:4], r[:4], atol=1e-4) and gt[4] == r[4]]
        assert len(match) == 1, 'box not recovered: %s' % (gt,)
    n_checked += len(bs)
print('PASS: encode -> decode round trip recovered all %d boxes exactly (atol=1e-4).' % n_checked)

Y_train = np.stack([encode(b) for b in BX_train])
Y_test  = np.stack([encode(b) for b in BX_test])
print('target tensor shape:', Y_train.shape, '= (N, S, S, 5 + C)')
print('cells with an object: %.2f%% of all cells  <- note how sparse the target already is'
      % (100 * Y_train[..., 4].mean()))

The assertion passes, and note the last line: **only about 5.6% of cells contain an object**, so ~94% of the target is background. Even in this tiny, deliberately dense toy problem the target is overwhelmingly negative. Hold that thought until Section 18.

Now the visual proof: original boxes (solid) versus boxes recovered from the encoded tensor (dashed, drawn on top). They coincide exactly.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for k, ax in enumerate(axes):
    ax.imshow(X_train[k])
    draw_grid(ax)
    draw_boxes(ax, BX_train[k], lw=3.0, ls='-')                 # original, thick
    rec = decode(encode(BX_train[k]))
    for (cx, cy, w, h, cls) in rec:                             # recovered, dashed white
        ax.add_patch(patches.Rectangle((cx - w/2, cy - h/2), w, h, fill=False,
                                       edgecolor='white', lw=1.2, ls=(0, (3, 3))))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title('%d box(es) recovered' % len(rec), fontsize=9)
plt.suptitle('Encode -> decode round trip: thick colour = ground truth, dashed white = decoded. '
             'They overlap exactly.', y=1.04)
plt.tight_layout(); plt.show()

---
# 6. YOLO v1: the loss function

This is the heart of YOLO v1. It is one big sum-squared-error, but every term is there for a reason:

$$
\begin{aligned}
\mathcal{L} = \;
& \lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbf{1}^{\text{obj}}_{ij}
  \left[ (x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 \right] & \text{(1) centre} \\[2pt]
+\; & \lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbf{1}^{\text{obj}}_{ij}
  \left[ \left(\sqrt{w_i} - \sqrt{\hat{w}_i}\right)^2 + \left(\sqrt{h_i} - \sqrt{\hat{h}_i}\right)^2 \right] & \text{(2) size} \\[2pt]
+\; & \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbf{1}^{\text{obj}}_{ij} \left( C_i - \hat{C}_i \right)^2 & \text{(3) confidence, object} \\[2pt]
+\; & \lambda_{\text{noobj}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbf{1}^{\text{noobj}}_{ij}
  \left( C_i - \hat{C}_i \right)^2 & \text{(4) confidence, no object} \\[2pt]
+\; & \sum_{i=0}^{S^2} \mathbf{1}^{\text{obj}}_{i} \sum_{c \in \text{classes}}
  \left( p_i(c) - \hat{p}_i(c) \right)^2 & \text{(5) classification}
\end{aligned}
$$

where:
- $\mathbf{1}^{\text{obj}}_{i}$ denotes if an object appears in cell $i$.
- $\mathbf{1}^{\text{obj}}_{ij}$ denotes that the $j^{\text{th}}$ bounding box predictor in cell $i$ is **responsible** for that prediction. Among the $B$ boxes in the responsible cell, the one with the **highest current IoU** with the ground truth is elected responsible. This is a form of self-organisation: the $B$ predictors specialise over training (one drifts towards tall boxes, another towards wide ones) without ever being told to.
- $\mathbf{1}^{\text{noobj}}_{ij}$ is the complement, $1 - \mathbf{1}^{\text{obj}}_{ij}$: every box that is not responsible for a ground truth object, which is nearly all of them.

## Why the two $\lambda$ constants?

- $\lambda_{\text{coord}} = 5$. Localisation is the job we actually care about, and sum-squared error weights it equally with classification by default. Turning it up says "getting the box right matters more".
- $\lambda_{\text{noobj}} = 0.5$. **Most cells contain no object.** In our data only ~5.6% of cells do. If every empty cell pushed its confidence to zero at full strength, that term would overwhelm the gradient from the few cells that do contain objects, and the model would collapse to predicting "nothing anywhere". Down-weighting it by 0.5 is YOLO's blunt instrument against class imbalance.
- Remember this: it is a hand-tuned constant patching over a structural problem. **Focal loss (Section 20) is the principled version of this same fix**, and RetinaNet exists because $\lambda_{\text{noobj}} = 0.5$ is not good enough.

## Why $\sqrt{w}$ and $\sqrt{h}$?

Sum-squared error treats an absolute error of 10 pixels the same whether the box is 300 pixels wide or 20 pixels wide. But a 10 pixel error on a 20 pixel box is a *disaster*, while on a 300 pixel box it is a rounding error. The square root partially compensates: it **compresses large values and stretches small ones**, so the same absolute deviation produces a larger loss on a small box. Let us see the curve.

In [ ]:
w = np.linspace(0.001, 1.0, 500)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].plot(w, w, color=MUTED, ls='--', label='identity:  w')
ax[0].plot(w, np.sqrt(w), color=C_BLUE, label=r'$\sqrt{w}$')
ax[0].set_xlabel('w (box width, fraction of image)'); ax[0].set_ylabel('encoded value')
ax[0].set_title(r'$\sqrt{\cdot}$ stretches small boxes, compresses large ones'); ax[0].legend()

delta = 0.05   # same absolute width error, applied to boxes of different size
sizes = np.linspace(0.05, 0.95, 300)
err_plain = (sizes - (sizes + delta))**2 * np.ones_like(sizes)
err_sqrt = (np.sqrt(sizes) - np.sqrt(sizes + delta))**2
ax[1].plot(sizes, err_plain / err_plain.max(), color=MUTED, ls='--', label='SSE on w (flat)')
ax[1].plot(sizes, err_sqrt / err_sqrt.max(), color=C_BLUE, label=r'SSE on $\sqrt{w}$')
ax[1].set_xlabel('true box width w'); ax[1].set_ylabel('loss (normalised)')
ax[1].set_title('Same +0.05 absolute error, different box sizes'); ax[1].legend()
ax[1].annotate('small boxes punished more', xy=(0.12, 0.92), xytext=(0.35, 0.75),
               color=C_BLUE, fontsize=9, arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.2))
plt.tight_layout(); plt.show()

for s_ in [0.06, 0.2, 0.6]:
    print('true w=%.2f, predicted w=%.2f  ->  SSE on w = %.5f   SSE on sqrt(w) = %.5f'
          % (s_, s_ + delta, delta**2, (np.sqrt(s_) - np.sqrt(s_ + delta))**2))

The printout says it cleanly: with plain SSE the loss for a 0.05 width error is `0.0025` regardless of box size. With the square root, the same 0.05 error costs **about 7.5x more** on a small box (w=0.06, loss 0.0075) than on a large one (w=0.6, loss 0.0010). It is a crude fix, and only a partial one: the compensation is milder than the true relative-error scaling. YOLO v2 replaces it outright with anchor boxes and a log-space parameterisation, but for v1 it is cheap and it helps.

## 6.1 IoU, the primitive everything depends on

IoU appears three times in YOLO: to elect the responsible box, as the confidence *target*, and inside NMS. We write it once, vectorised over arbitrary leading dimensions, using corner coordinates in normalised image units.

In [ ]:
# Grid index tensors, used to convert cell-relative (x, y) into image-relative centres.
_gi, _gj = torch.meshgrid(torch.arange(S), torch.arange(S), indexing='ij')
GI = _gi.float().to(device)     # row index  (i) of each cell
GJ = _gj.float().to(device)     # col index  (j) of each cell

def to_corners(box, gi, gj):
    "(..., 4) cell-relative (x, y, w, h) -> (..., 4) corners (x1, y1, x2, y2) in [0,1] image units."
    cx = (gj + box[..., 0]) / S          # x is relative to the cell -> add cell index, scale by S
    cy = (gi + box[..., 1]) / S
    w, h = box[..., 2], box[..., 3]      # w, h already relative to the whole image
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)

def iou(a, b):
    "Elementwise IoU between two broadcastable sets of corner boxes."
    x1 = torch.max(a[..., 0], b[..., 0]); y1 = torch.max(a[..., 1], b[..., 1])
    x2 = torch.min(a[..., 2], b[..., 2]); y2 = torch.min(a[..., 3], b[..., 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_a = (a[..., 2] - a[..., 0]).clamp(min=0) * (a[..., 3] - a[..., 1]).clamp(min=0)
    area_b = (b[..., 2] - b[..., 0]).clamp(min=0) * (b[..., 3] - b[..., 1]).clamp(min=0)
    return inter / (area_a + area_b - inter + 1e-9)

# sanity check against the exercise at the end of this notebook:
# box A is 2x2, box B is 2x3, overlap is 1x1  ->  IoU = 1 / (4 + 6 - 1) = 1/9
A = torch.tensor([0., 0., 2., 2.])            # area 4
Bx = torch.tensor([1., 1., 3., 4.])           # area 2*3 = 6, intersection = 1x1 = 1
print('IoU check: %.4f  (expected 1/9 = %.4f)' % (iou(A, Bx).item(), 1/9))
assert abs(iou(A, Bx).item() - 1/9) < 1e-6

## 6.2 The five loss terms, implemented separately

Each term below maps one-to-one onto a line of the equation. We return them **individually** as well as summed, so we can watch each one during training and see which part of the problem the network is solving at any moment.

Two implementation details worth naming:
- `.detach()` on the IoU used as the confidence target: it is a *target*, so no gradient flows through it. Forgetting this lets the network cheat by shrinking boxes to game the IoU.
- `.clamp(min=1e-6)` before `sqrt`: the derivative of $\sqrt{x}$ is $1/(2\sqrt{x})$, which blows up at $x=0$. Predicted widths start near zero, so this guard is not optional.

In [ ]:
def yolo_v1_loss(pred_box, pred_cls, target, lambda_coord=5.0, lambda_noobj=0.5):
    '''
    pred_box : (N, S, S, B, 5)  -> x, y, w, h, confidence   (all already in [0,1])
    pred_cls : (N, S, S, C)     -> class probabilities (softmax)
    target   : (N, S, S, 5 + C) -> x, y, w, h, objectness, one-hot class
    Returns (total, terms) with terms = [xy, wh, obj, noobj, cls], all per-image means.
    '''
    N = pred_box.shape[0]
    gi = GI[None, :, :, None]                       # (1, S, S, 1) broadcast over batch and B
    gj = GJ[None, :, :, None]

    t_box = target[..., :4]                         # (N, S, S, 4)
    t_obj = target[..., 4]                          # (N, S, S)  1 if this cell owns an object
    t_cls = target[..., 5:]                         # (N, S, S, C)

    # ---- elect the responsible box j: the one with highest IoU with the ground truth ----
    pred_corners = to_corners(pred_box, gi, gj)                       # (N, S, S, B, 4)
    t_corners = to_corners(t_box, GI[None], GJ[None])                 # (N, S, S, 4)
    t_corners = t_corners[:, :, :, None, :].expand_as(pred_corners)   # (N, S, S, B, 4)
    ious = iou(pred_corners, t_corners).detach()                      # (N, S, S, B), a TARGET
    best = ious.argmax(dim=-1, keepdim=True)
    responsible = torch.zeros_like(ious).scatter_(-1, best, 1.0)      # one-hot over B

    obj_ij = responsible * t_obj[..., None]         # 1^obj_ij : responsible box in an object cell
    noobj_ij = 1.0 - obj_ij                         # 1^noobj_ij : everything else

    t_box_e = t_box[:, :, :, None, :].expand(-1, -1, -1, B, -1)

    # (1) centre: penalise (x, y) of the responsible box only
    l_xy = lambda_coord * (obj_ij * ((pred_box[..., 0] - t_box_e[..., 0])**2 +
                                     (pred_box[..., 1] - t_box_e[..., 1])**2)).sum()

    # (2) size: SSE in sqrt space, so equal absolute error costs more on small boxes
    l_wh = lambda_coord * (obj_ij * ((pred_box[..., 2].clamp(min=1e-6).sqrt() - t_box_e[..., 2].sqrt())**2 +
                                     (pred_box[..., 3].clamp(min=1e-6).sqrt() - t_box_e[..., 3].sqrt())**2)).sum()

    # (3) confidence where an object IS: target C_i = Pr(Object) * IoU = 1 * IoU
    conf_target = ious * t_obj[..., None]
    l_obj = (obj_ij * (pred_box[..., 4] - conf_target)**2).sum()

    # (4) confidence where no object is: target 0, down-weighted by lambda_noobj
    l_noobj = lambda_noobj * (noobj_ij * (pred_box[..., 4] - 0.0)**2).sum()

    # (5) classification: one set of class probs per cell, only for cells with an object
    l_cls = (t_obj * ((pred_cls - t_cls)**2).sum(-1)).sum()

    terms = torch.stack([l_xy, l_wh, l_obj, l_noobj, l_cls]) / N
    return terms.sum(), terms

TERM_NAMES = ['xy (centre)', 'wh (size)', 'obj conf', 'noobj conf', 'class']
TERM_COLORS = [C_BLUE, C_AQUA, C_YELLOW, C_VIOLET, C_RED]
print('loss defined. terms:', TERM_NAMES)

## 6.3 The model: a tiny backbone and a detection head

The architecture mirrors full YOLO, just smaller. A **backbone** that downsamples $96 \times 96$ down to our $6 \times 6$ grid (four stride-2 stages: 96 to 48 to 24 to 12 to 6), then a **head** that maps features to the $B \cdot 5 + C = 13$ output channels.

The key structural insight: **the grid is not something we impose afterwards, it is just the spatial resolution of the last feature map.** A $6 \times 6 \times 13$ output tensor *is* a conv feature map with 13 channels. That is why single-stage detection is fast: it is one forward pass of a plain CNN.

Output activations:
- `sigmoid` on $(x, y)$ so the predicted centre stays **inside its own cell**, which is exactly the constraint YOLO v2 later formalised (Section 8). The original v1 used a linear output here, and paid for it with unstable early training.
- `sigmoid` on $(w, h)$ so sizes stay a valid fraction of the image, and on confidence so it is a valid probability-like score.
- `softmax` on the class channels, then **sum-squared error** against the one-hot target, faithful to the v1 loss above (v3 later swaps this for independent logistic classifiers).
- We use BatchNorm, which v1 did not have and v2 added. It is doing real work in making this converge in a few hundred steps.

In [ ]:
class TinyYOLO(nn.Module):
    def __init__(self):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.LeakyReLU(0.1),
                nn.MaxPool2d(2))
        # BACKBONE: 96 -> 48 -> 24 -> 12 -> 6, so the final feature map IS the S x S grid
        self.backbone = nn.Sequential(block(3, 16), block(16, 32), block(32, 64), block(64, 128))
        # HEAD: mix features, then project to B*5 + C channels
        self.head = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.1),
            nn.Conv2d(128, B * 5 + C, 1))

    def forward(self, x):
        feat = self.backbone(x)                       # (N, 128, S, S)
        out = self.head(feat).permute(0, 2, 3, 1)     # (N, S, S, B*5+C)
        box = torch.sigmoid(out[..., :B*5].reshape(-1, S, S, B, 5))   # x,y,w,h,conf in [0,1]
        cls = torch.softmax(out[..., B*5:], dim=-1)                   # class probabilities
        return box, cls

model = TinyYOLO().to(device)
n_params = sum(p.numel() for p in model.parameters())

_x = torch.zeros(2, 3, IMG, IMG, device=device)
with torch.no_grad():
    _f = model.backbone(_x); _b, _c = model(_x)
print('parameters: %s' % format(n_params, ','))
print('input          ', tuple(_x.shape))
print('backbone output', tuple(_f.shape), '  <- 128 channels on the %dx%d grid' % (S, S))
print('boxes          ', tuple(_b.shape), '  = (N, S, S, B, 5)')
print('classes        ', tuple(_c.shape), '  = (N, S, S, C)')
print('flattened head output = %d x %d x %d, exactly the S x S x (B*5+C) tensor from Section 4'
      % (S, S, B*5+C))

### Slider: what $\lambda_{\text{coord}}$ and $\lambda_{\text{noobj}}$ actually do to the loss

Below we take one real minibatch through the **untrained** network and show how each loss term contributes as you change the two constants. Things to try:

- Set $\lambda_{\text{noobj}} = 1$ (its unweighted value). The `noobj` term dwarfs everything, because ~95% of cells are empty. That imbalance is what the 0.5 is fighting.
- Set $\lambda_{\text{coord}} = 1$. The localisation terms shrink to near-invisibility next to the confidence terms, and the network has little incentive to place boxes well.
- The paper's choice, $(5, 0.5)$, is a compromise found by hand.

In [ ]:
Xb_demo = torch.from_numpy(X_train[:32]).permute(0, 3, 1, 2).to(device)
Yb_demo = torch.from_numpy(Y_train[:32]).to(device)
with torch.no_grad():
    _pb_demo, _pc_demo = model(Xb_demo)

def show_lambda(lambda_coord=5.0, lambda_noobj=0.5):
    with torch.no_grad():
        total, terms = yolo_v1_loss(_pb_demo, _pc_demo, Yb_demo, lambda_coord, lambda_noobj)
    vals = terms.cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [1.3, 1]})
    bars = ax[0].bar(TERM_NAMES, vals, color=TERM_COLORS, width=0.62)
    for b_, v in zip(bars, vals):
        ax[0].text(b_.get_x() + b_.get_width()/2, v, ' %.2f' % v, ha='center', va='bottom',
                   fontsize=9, color=INK2)
    ax[0].set_ylabel('loss contribution'); ax[0].set_ylim(0, max(vals.max()*1.25, 1e-3))
    ax[0].set_title('Per-term contribution (untrained net, one batch)')
    ax[0].tick_params(axis='x', labelrotation=12)

    share = 100 * vals / vals.sum()
    left = 0.0
    for v, n_, c_ in zip(share, TERM_NAMES, TERM_COLORS):
        ax[1].barh([0], [v], left=left, color=c_, height=0.5, edgecolor='white', linewidth=2)
        if v > 6:
            ax[1].text(left + v/2, 0, '%.0f%%' % v, ha='center', va='center',
                       color='white', fontsize=9, weight='bold')
        left += v
    ax[1].set_xlim(0, 100); ax[1].set_yticks([]); ax[1].grid(False)
    ax[1].set_xlabel('share of total loss (%)')
    ax[1].set_title('total = %.2f    (lc=%.1f, ln=%.2f)' % (vals.sum(), lambda_coord, lambda_noobj))
    handles = [patches.Patch(color=c_, label=n_) for c_, n_ in zip(TERM_COLORS, TERM_NAMES)]
    ax[1].legend(handles=handles, fontsize=7.5, ncol=2, loc='lower center', bbox_to_anchor=(0.5, -0.62))
    plt.tight_layout(); plt.show()

interact(show_lambda,
         lambda_coord=widgets.FloatSlider(value=5.0, min=0.5, max=10.0, step=0.5,
                                          description='lambda_coord', continuous_update=False,
                                          style={'description_width': '110px'}),
         lambda_noobj=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                          description='lambda_noobj', continuous_update=False,
                                          style={'description_width': '110px'}));

## 6.4 Training

Standard loop: Adam, cosine decay, 300 steps of batch size 32 (about 18 passes over our 512 images). This takes well under a minute on a Colab CPU.

First, let us see what the **untrained** network predicts, so the comparison afterwards is honest.

In [ ]:
X_test_t = torch.from_numpy(X_test).permute(0, 3, 1, 2).to(device)
Y_test_t = torch.from_numpy(Y_test).to(device)
X_train_t = torch.from_numpy(X_train).permute(0, 3, 1, 2).to(device)
Y_train_t = torch.from_numpy(Y_train).to(device)

def raw_boxes_for_image(pb, pc, k, conf_thresh):
    "Every box above threshold, WITHOUT nms. Returns (cx, cy, w, h, score, cls) in pixels."
    dets = []
    for i in range(S):
        for j in range(S):
            cls = int(np.argmax(pc[k, i, j])); cprob = pc[k, i, j, cls]
            for b in range(B):
                x, y, w, h, conf = pb[k, i, j, b]
                score = conf * cprob            # Pr(Class) * IoU, from Section 5
                if score > conf_thresh:
                    dets.append([(j + x) * CELL, (i + y) * CELL, w * IMG, h * IMG, float(score), cls])
    return dets

@torch.no_grad()
def predict_numpy(X_t):
    model.eval()
    pb, pc = model(X_t)
    return pb.cpu().numpy(), pc.cpu().numpy()

pb_before, pc_before = predict_numpy(X_test_t)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for k, ax in enumerate(axes):
    ax.imshow(X_test[k])
    for d in raw_boxes_for_image(pb_before, pc_before, k, 0.0):
        cx, cy, w, h, sc, cl = d
        ax.add_patch(patches.Rectangle((cx-w/2, cy-h/2), w, h, fill=False,
                                       edgecolor=C_RED, lw=0.7, alpha=0.5))
    draw_boxes(ax, BX_test[k], lw=2.0)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_xlim(0, IMG); ax.set_ylim(IMG, 0)
plt.suptitle('BEFORE training: all 72 predicted boxes (red) are junk, roughly half the image, everywhere',
             y=1.03)
plt.tight_layout(); plt.show()
print('untrained confidence: min %.3f, max %.3f (essentially uninformative)'
      % (pb_before[..., 4].min(), pb_before[..., 4].max()))

In [ ]:
# We train the SAME model instance whose untrained predictions we just plotted above,
# so the before/after comparison is honest.
STEPS, BATCH = 300, 32
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
gen = torch.Generator().manual_seed(3)

history = []
model.train()
t0 = time.time()
for step in range(STEPS):
    idx = torch.randint(0, X_train_t.shape[0], (BATCH,), generator=gen)
    pb, pc = model(X_train_t[idx])
    loss, terms = yolo_v1_loss(pb, pc, Y_train_t[idx])
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    history.append([loss.item()] + terms.detach().cpu().tolist())
    if step % 50 == 0 or step == STEPS - 1:
        print('step %3d  loss %6.3f   xy %.3f  wh %.3f  obj %.3f  noobj %.3f  cls %.3f'
              % (step, loss.item(), *terms.detach().cpu().tolist()))
history = np.array(history)
print('\ntrained %d steps in %.1fs on %s' % (STEPS, time.time() - t0, device))
print('mean loss over first 10 steps: %.3f' % history[:10, 0].mean())
print('mean loss over last  10 steps: %.3f' % history[-10:, 0].mean())
print('reduction: %.1fx' % (history[:10, 0].mean() / history[-10:, 0].mean()))

## 6.5 Loss curves, per term

The total is not very informative on its own. The per-term breakdown tells the actual story of what the network learned and in what order.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(history[:, 0], color=C_BLUE)
ax[0].set_yscale('log'); ax[0].set_xlabel('step'); ax[0].set_ylabel('total loss (log scale)')
ax[0].set_title('Total loss: %.2f -> %.3f' % (history[:10, 0].mean(), history[-10:, 0].mean()))

def smooth(a, k=9):
    return np.convolve(a, np.ones(k)/k, mode='valid')
for t in range(5):
    ax[1].plot(smooth(history[:, 1 + t]), color=TERM_COLORS[t], label=TERM_NAMES[t])
ax[1].set_yscale('log'); ax[1].set_xlabel('step'); ax[1].set_ylabel('term value (log, smoothed)')
ax[1].set_title('Per-term losses'); ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

print('final per-term values (mean of last 10 steps):')
for t in range(5):
    print('   %-12s %.4f' % (TERM_NAMES[t], history[-10:, 1 + t].mean()))

Read the right-hand panel carefully, it is a nice illustration of loss dynamics:

- **`noobj` collapses first.** The cheapest possible win is to shut off confidence everywhere. This is the network taking the path of least resistance, and it is exactly the degenerate behaviour that $\lambda_{\text{noobj}}$ and later focal loss are managing.
- **`class` vanishes almost immediately.** Our classes are colour-coded, so this is nearly free. On real data it would not be.
- **`xy` and `wh` fall steadily**, and are what $\lambda_{\text{coord}} = 5$ keeps alive.
- **`obj` plateaus highest.** That is expected and is *not* a bug: its target is the live IoU, a moving goalpost that rises as the boxes improve. The network is chasing its own accuracy.

## 6.6 Decode, threshold, and NMS

The network emits $S \times S \times B = 72$ boxes for every image. Turning that into a clean set of detections takes three steps:

1. **Decode**: cell-relative $(x, y)$ and image-relative $(w, h)$ back into pixels (the same `decode` maths we asserted in Section 5.1).
2. **Threshold** on the class-specific confidence $\Pr(\text{Class}_i) \times \text{IoU}$.
3. **Non-max suppression**: sort by score, keep the best box, discard anything overlapping it by more than an IoU threshold, repeat. Applied **per class**, so a rectangle never suppresses an overlapping ellipse.

NMS is the one piece of YOLO that is *not* learned, it is a greedy post-process. We write it from scratch and then cross-check it against `torchvision.ops.nms` to prove our version is correct.

In [ ]:
def box_iou_xywh(p, q):
    "IoU between two (cx, cy, w, h) boxes in pixels."
    ax1, ay1, ax2, ay2 = p[0]-p[2]/2, p[1]-p[3]/2, p[0]+p[2]/2, p[1]+p[3]/2
    bx1, by1, bx2, by2 = q[0]-q[2]/2, q[1]-q[3]/2, q[0]+q[2]/2, q[1]+q[3]/2
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    return inter / ((ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter + 1e-9)

def nms(dets, iou_thresh=0.4):
    "Greedy per-class NMS. dets: list of (cx, cy, w, h, score, cls). Returns the survivors."
    keep = []
    for cls in sorted(set(int(d[5]) for d in dets)):
        cand = sorted([d for d in dets if int(d[5]) == cls], key=lambda z: -z[4])
        while cand:
            best = cand.pop(0)
            keep.append(best)
            cand = [q for q in cand if box_iou_xywh(best, q) < iou_thresh]   # suppress overlaps
    return keep

# ---- cross-check our NMS against torchvision on random boxes ----
from torchvision.ops import nms as tv_nms
rng_n = np.random.default_rng(7)
_b = rng_n.uniform(10, 80, size=(60, 2)); _wh = rng_n.uniform(10, 40, size=(60, 2))
_sc = rng_n.uniform(0, 1, size=60)
mine = nms([[_b[i,0], _b[i,1], _wh[i,0], _wh[i,1], _sc[i], 0] for i in range(60)], 0.4)
mine_ids = sorted([int(np.argmin(np.abs(_sc - m[4]))) for m in mine])
tvb = torch.tensor(np.stack([_b[:,0]-_wh[:,0]/2, _b[:,1]-_wh[:,1]/2,
                             _b[:,0]+_wh[:,0]/2, _b[:,1]+_wh[:,1]/2], 1), dtype=torch.float32)
tv_ids = sorted(tv_nms(tvb, torch.tensor(_sc, dtype=torch.float32), 0.4).tolist())
print('our nms kept   :', mine_ids)
print('torchvision    :', tv_ids)
assert mine_ids == tv_ids, 'our NMS disagrees with torchvision'
print('PASS: our from-scratch NMS matches torchvision.ops.nms exactly (%d of 60 boxes kept).'
      % len(tv_ids))

### After training: the confidence heatmap

The heatmap shows $\max_j \hat{C}_{ij}$ per cell, that is, how strongly each cell believes it owns an object. Compare it to where the object centres actually are.

In [ ]:
pb_after, pc_after = predict_numpy(X_test_t)

fig, axes = plt.subplots(2, 5, figsize=(13.5, 5.6))
for k in range(5):
    ax = axes[0, k]
    ax.imshow(X_test[k]); draw_grid(ax); draw_boxes(ax, BX_test[k], lw=1.6)
    for (cx, cy, _w, _h, _c) in BX_test[k]:
        ax.plot([cx], [cy], marker='o', ms=4, color='white', mec=INK, mew=0.8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title('ground truth', fontsize=9)

    ax = axes[1, k]
    heat = pb_after[k, :, :, :, 4].max(axis=-1)          # max confidence over the B boxes
    im = ax.imshow(heat, cmap=CMAP_BLUE, vmin=0, vmax=1, extent=[0, IMG, IMG, 0],
                   interpolation='nearest')
    for (cx, cy, _w, _h, _c) in BX_test[k]:
        ax.plot([cx], [cy], marker='o', ms=6, color='white', mec=INK, mew=1.2)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title('confidence per cell', fontsize=9)
fig.colorbar(im, ax=axes[1, :].tolist(), fraction=0.02, pad=0.01, label='max confidence')
plt.suptitle('The network lights up exactly the cells containing object centres (white dots)', y=0.99)
plt.show()

### Seeing NMS actually do something

An honest caveat about this demo. At our operating threshold of 0.30 the detector is so clean on this easy synthetic task that **NMS has almost nothing to suppress**: it removes only ~23 boxes out of ~155 across the whole test set, and usually zero per image. A "before and after NMS" figure at that threshold would show two identical rows and teach you nothing.

So below we deliberately drop the confidence threshold to **0.05**, which is much closer to what a real detector's raw output looks like: many overlapping candidates stacked on every object, several cells firing on the same shape. That is the regime NMS was designed for, and now you can watch it work. We show the five test images where it does the most work.

- **Top row**: raw boxes above the low threshold. Note the duplicates piled on each object.
- **Bottom row**: what survives greedy per-class NMS at IoU > 0.4.

In [ ]:
NMS_T = 0.40
# Deliberately LOW threshold: at our operating point the detector is so clean that NMS has
# almost nothing to suppress, which would make for a useless demo. See the note below.
CONF_T_LOW = 0.05

# Pick the images where NMS actually does the most work, so the figure is informative.
supp = []
for k in range(len(BX_test)):
    r = raw_boxes_for_image(pb_after, pc_after, k, CONF_T_LOW)
    supp.append((len(r) - len(nms(r, NMS_T)), k))
show_ids = [k for _, k in sorted(supp, reverse=True)[:5]]

fig, axes = plt.subplots(2, 5, figsize=(13.5, 5.6))
for col, k in enumerate(show_ids):
    raw = raw_boxes_for_image(pb_after, pc_after, k, CONF_T_LOW)
    kept = nms(raw, NMS_T)
    for row, (dets, title) in enumerate([(raw, 'raw: %d boxes' % len(raw)),
                                         (kept, 'after NMS: %d boxes' % len(kept))]):
        ax = axes[row, col]
        ax.imshow(X_test[k])
        draw_boxes(ax, BX_test[k], lw=2.6, ls=(0, (2, 2)))       # ground truth, dashed
        # labels only on the cleaned row: on the raw row they would overlap into mush,
        # which is itself a fair illustration of why NMS is needed.
        draw_boxes(ax, dets, lw=1.4, label_scores=(row == 1))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_xlim(0, IMG); ax.set_ylim(IMG, 0)
        ax.set_title(title, fontsize=9)
plt.suptitle('NMS at a LOW confidence threshold (%.2f), where duplicates really exist. '
             'Top: raw. Bottom: after NMS at IoU > %.2f. Dashed = ground truth.'
             % (CONF_T_LOW, NMS_T), y=0.99)
plt.tight_layout(); plt.show()

for t in [CONF_T_LOW, 0.30]:
    raw_tot = sum(len(raw_boxes_for_image(pb_after, pc_after, k, t)) for k in range(len(BX_test)))
    nms_tot = sum(len(nms(raw_boxes_for_image(pb_after, pc_after, k, t), NMS_T))
                  for k in range(len(BX_test)))
    print('conf > %.2f : %4d raw boxes -> %4d after NMS  (%3d suppressed) over the %d test images'
          % (t, raw_tot, nms_tot, raw_tot - nms_tot, len(BX_test)))

### The final detections, at the operating threshold

And here is the payoff: the full pipeline (decode, threshold at 0.30, per-class NMS) on unseen test images. Solid boxes are our detections with their class-specific confidence; dashed boxes are ground truth.

In [ ]:
CONF_T = 0.30
fig, axes = plt.subplots(1, 6, figsize=(15, 2.9))
for col, k in enumerate(range(6)):
    dets = nms(raw_boxes_for_image(pb_after, pc_after, k, CONF_T), NMS_T)
    ax = axes[col]
    ax.imshow(X_test[k])
    draw_boxes(ax, BX_test[k], lw=2.8, ls=(0, (2, 2)))
    draw_boxes(ax, dets, lw=1.4, label_scores=True)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_xlim(0, IMG); ax.set_ylim(IMG, 0)
    ax.set_title('%d detected / %d GT' % (len(dets), len(BX_test[k])), fontsize=9)
plt.suptitle('Final detections on unseen test images (conf > %.2f, NMS IoU > %.2f). '
             'Solid = predicted, dashed = ground truth.' % (CONF_T, NMS_T), y=1.06)
plt.tight_layout(); plt.show()

### Did it actually work? A quantitative check

Visual inspection is not enough. Let us match detections to ground truth (correct class, IoU > 0.5) over the whole test set and report precision, recall, and the mean IoU of matched boxes.

In [ ]:
def evaluate(pb, pc, conf_thresh=0.30, nms_thresh=0.40, iou_match=0.5):
    tp = 0; n_gt = 0; n_pred = 0; matched_ious = []
    for k in range(len(BX_test)):
        dets = nms(raw_boxes_for_image(pb, pc, k, conf_thresh), nms_thresh)
        n_gt += len(BX_test[k]); n_pred += len(dets)
        used = set()
        for gt in BX_test[k]:
            best_v, best_i = -1.0, -1
            for di, d in enumerate(dets):
                if di in used or int(d[5]) != gt[4]:
                    continue
                v = box_iou_xywh(d, gt)
                if v > best_v:
                    best_v, best_i = v, di
            if best_v >= iou_match:
                tp += 1; used.add(best_i); matched_ious.append(best_v)
    prec = tp / max(n_pred, 1); rec = tp / max(n_gt, 1)
    f1 = 2*prec*rec/max(prec+rec, 1e-9)
    return dict(tp=tp, n_gt=n_gt, n_pred=n_pred, precision=prec, recall=rec, f1=f1,
                mean_iou=float(np.mean(matched_ious)) if matched_ious else 0.0)

r_before = evaluate(pb_before, pc_before)
r_after = evaluate(pb_after, pc_after)
print('%-22s %8s %8s' % ('', 'BEFORE', 'AFTER'))
for key in ['n_gt', 'n_pred', 'tp', 'precision', 'recall', 'f1', 'mean_iou']:
    fmt = '%8.3f' if isinstance(r_after[key], float) else '%8d'
    print(('%-22s ' + fmt + ' ' + fmt) % (key, r_before[key], r_after[key]))
print('\n(tp = detections with the correct class and IoU > 0.5 against a ground truth box)')
assert r_after['recall'] > 0.85 and r_after['precision'] > 0.85, 'the detector failed to converge'
print('\nPASS: the mini-YOLO detects the shapes at recall %.3f / precision %.3f, mean IoU %.3f.'
      % (r_after['recall'], r_after['precision'], r_after['mean_iou']))

### Slider: confidence threshold and NMS IoU threshold on the real trained model

These two numbers are the entire post-processing stage of a detector, and they trade off precision against recall. Try:

- **Confidence near 0**: every one of the 72 boxes survives thresholding, and NMS has to clean up the mess. Precision collapses.
- **Confidence near 1**: only the surest detections survive, and objects start being missed. Recall collapses.
- **NMS IoU near 1.0**: nothing is ever suppressed, so you see duplicate boxes stacked on each object.
- **NMS IoU near 0**: a box suppresses anything it touches at all, so two nearby objects of the same class lose one detection.

In [ ]:
def show_detections(image_id=0, conf_thresh=0.30, nms_iou=0.40):
    raw = raw_boxes_for_image(pb_after, pc_after, image_id, conf_thresh)
    kept = nms(raw, nms_iou)
    r = evaluate(pb_after, pc_after, conf_thresh, nms_iou)
    fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.9))
    for a, dets, title in [(ax[0], raw, 'thresholded: %d boxes' % len(raw)),
                           (ax[1], kept, 'after NMS: %d boxes' % len(kept))]:
        a.imshow(X_test[image_id])
        draw_boxes(a, BX_test[image_id], lw=2.6, ls=(0, (2, 2)))
        draw_boxes(a, dets, lw=1.4, label_scores=True)
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
        a.set_xlim(0, IMG); a.set_ylim(IMG, 0); a.set_title(title, fontsize=10)
    ax[2].bar(['precision', 'recall', 'F1'], [r['precision'], r['recall'], r['f1']],
              color=[C_BLUE, C_AQUA, C_VIOLET], width=0.6)
    for i_, v_ in enumerate([r['precision'], r['recall'], r['f1']]):
        ax[2].text(i_, v_, ' %.3f' % v_, ha='center', va='bottom', fontsize=9, color=INK2)
    ax[2].set_ylim(0, 1.15); ax[2].set_title('whole test set (%d GT objects)' % r['n_gt'], fontsize=10)
    plt.tight_layout(); plt.show()

interact(show_detections,
         image_id=widgets.IntSlider(value=0, min=0, max=15, step=1, description='image',
                                    continuous_update=False),
         conf_thresh=widgets.FloatSlider(value=0.30, min=0.0, max=0.95, step=0.05,
                                         description='confidence', continuous_update=False,
                                         style={'description_width': '90px'}),
         nms_iou=widgets.FloatSlider(value=0.40, min=0.05, max=1.0, step=0.05,
                                     description='NMS IoU', continuous_update=False,
                                     style={'description_width': '90px'}));

---
# 7. YOLO v1: limitations

Straight from the paper, and every one of them traces back to a design decision we just implemented:

- **Detects only a small number of objects.** At most $S^2 = 49$ in the paper (36 for us), because each cell emits one class and effectively one object.
- **Cannot detect objects that are small or close together, due to strong spatial constraints.** Two object centres in one cell is a target the output tensor *cannot represent*. Remember we explicitly rejection-sampled our data to avoid this, precisely because the architecture cannot express it. A flock of birds is the classic failure case.
- **High localisation error.** Boxes are regressed from scratch with no priors, so the network must learn the whole space of shapes from a random init. The $\sqrt{w}$ trick is a partial patch.
- **Relatively low recall** compared to region-proposal methods.

The fixes for these are exactly what the next versions are:

| Limitation | Fixed by |
|---|---|
| One class and one box per cell | **anchor boxes** (v2), which decouple predictions per cell |
| Boxes regressed from nothing | **anchor priors from k-means** (v2), regress an *offset* from a sensible prior |
| Unstable early box predictions | **sigmoid-constrained centres** (v2) |
| Small objects missed | **multi-scale prediction** (v3), **FPN** |
| Low recall, imbalance | **focal loss** (RetinaNet) |

---
# 8. YOLO v2: anchor boxes

Instead of regressing box coordinates from nothing, v2 starts from a set of **anchor box priors** (also called default or prior boxes) and predicts a *correction* to each. The network predicts **5 coordinates per anchor box**: $t_x, t_y, t_w, t_h, t_o$.

If the cell is offset from the top left corner of the image by $(c_x, c_y)$, and the anchor box prior has width and height $(p_w, p_h)$, then the predictions correspond to:

$$
\begin{aligned}
b_x &= \sigma(t_x) + c_x \\
b_y &= \sigma(t_y) + c_y \\
b_w &= p_w\, e^{t_w} \\
b_h &= p_h\, e^{t_h} \\
\Pr(\text{object}) \times \text{IOU}(b, \text{object}) &= \sigma(t_o)
\end{aligned}
$$

Why each piece is the way it is:

- **$\sigma$ on the centre.** The sigmoid squashes $t_x$ into $(0, 1)$, so $b_x$ can only ever land **inside the cell that predicted it**. Without it (v1 used a direct linear prediction) any cell can propose a box centred anywhere in the image, and early in training every cell fights over every object. The paper reports this instability explicitly; the constraint makes the parameterisation easier to learn and **stabilises early training**. This is the same trick we already used in our v1 head.
- **$\exp$ on the size.** $e^{t_w} > 0$ always, so the width cannot go negative, and the prediction is *multiplicative*: $t_w = 0$ means "exactly the anchor prior", $t_w = 0.69$ means "twice the prior". Regressing in log space makes a factor-of-2 error cost the same whether the box is big or small. That is the same concern the $\sqrt{\cdot}$ trick addressed in v1, handled properly.
- **$\sigma$ on objectness**, keeping the confidence a valid score, with the same meaning as v1.

The network's job changes from "guess the box" to "**nudge the nearest sensible prior**", which is a far easier function to learn.

In [ ]:
def yolo_v2_decode(t_x, t_y, t_w, t_h, c_x, c_y, p_w, p_h):
    '''YOLO v2 box decode. Cell coords (c_x, c_y) in grid units; (p_w, p_h) anchor prior in
    grid units. Returns box centre and size in grid units.'''
    b_x = 1 / (1 + np.exp(-t_x)) + c_x        # sigma(t_x) + c_x  -> stays inside the cell
    b_y = 1 / (1 + np.exp(-t_y)) + c_y
    b_w = p_w * np.exp(t_w)                   # multiplicative correction to the prior
    b_h = p_h * np.exp(t_h)
    return b_x, b_y, b_w, b_h

# t = 0 must reproduce the anchor prior, centred in the middle of its cell
bx_, by_, bw_, bh_ = yolo_v2_decode(0, 0, 0, 0, c_x=2, c_y=3, p_w=1.5, p_h=2.0)
print('t = (0,0,0,0) with cell (2,3), prior 1.5 x 2.0 grid units:')
print('   centre (%.2f, %.2f)  <- exactly the cell centre' % (bx_, by_))
print('   size   (%.2f, %.2f)  <- exactly the prior' % (bw_, bh_))
assert abs(bx_ - 2.5) < 1e-9 and abs(bw_ - 1.5) < 1e-9
print('\nsigma bounds the centre offset to (0, 1), so b_x is always within [c_x, c_x + 1]:')
for t in [-6, -2, 0, 2, 6]:
    print('   t_x = %+d  ->  sigma(t_x) = %.4f  ->  b_x = %.4f' % (t, 1/(1+np.exp(-t)), 1/(1+np.exp(-t)) + 2))

### Slider: the sigmoid/exp decode geometry

The dotted box is the **anchor prior** $(p_w, p_h)$ sitting in its cell. The solid box is the **decoded prediction** $b$. Drag the $t$ values and watch:

- $t_x, t_y$ slide the centre, but the centre **can never leave the shaded cell**, no matter how extreme you make them. That is the sigmoid doing its job.
- $t_w, t_h$ scale the box **multiplicatively** around the prior. At $t=0$ the prediction *is* the prior.
- Compare $t_w = -1$ and $t_w = +1$: they scale by $1/e$ and $e$, symmetric factors. In log space, "half as wide" and "twice as wide" are equidistant from the prior, which is exactly the symmetry plain $w$ regression lacks.

In [ ]:
def show_v2_decode(t_x=0.0, t_y=0.0, t_w=0.0, t_h=0.0, p_w=1.5, p_h=2.0):
    c_x, c_y = 2, 3                                    # which cell we are in
    b_x, b_y, b_w, b_h = yolo_v2_decode(t_x, t_y, t_w, t_h, c_x, c_y, p_w, p_h)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.15, 1]})

    a = ax[0]
    for g in range(7):
        a.axhline(g, color=GRIDC, lw=0.8); a.axvline(g, color=GRIDC, lw=0.8)
    a.add_patch(patches.Rectangle((c_x, c_y), 1, 1, facecolor=C_BLUE, alpha=0.12,
                                  edgecolor=C_BLUE, lw=1.2))                 # the cell
    a.add_patch(patches.Rectangle((c_x + 0.5 - p_w/2, c_y + 0.5 - p_h/2), p_w, p_h, fill=False,
                                  edgecolor=MUTED, lw=1.6, ls=(0, (3, 3))))  # anchor prior
    a.add_patch(patches.Rectangle((b_x - b_w/2, b_y - b_h/2), b_w, b_h, fill=False,
                                  edgecolor=C_RED, lw=2.2))                  # decoded box
    a.plot([b_x], [b_y], marker='o', ms=7, color=C_RED, mec='white', mew=1.2)
    a.annotate('', xy=(b_x, b_y), xytext=(c_x, c_y),
               arrowprops=dict(arrowstyle='->', color=C_RED, lw=1.0, alpha=0.7))
    a.text(c_x - 0.05, c_y - 0.12, r'$(c_x, c_y)$', fontsize=9, color=C_BLUE, ha='right')
    a.text(0.05, 6.75, 'dotted grey = anchor prior $(p_w, p_h)$', fontsize=8.5, color=MUTED)
    a.text(0.05, 6.45, 'red = decoded box $b$', fontsize=8.5, color=C_RED)
    a.text(0.05, 6.15, 'blue cell = the only place the centre can be', fontsize=8.5, color=C_BLUE)
    a.set_xlim(0, 7); a.set_ylim(7, 0); a.set_aspect('equal'); a.grid(False)
    a.set_xticks(range(8)); a.set_yticks(range(8))
    a.set_title('decode in grid units', fontsize=10)

    a = ax[1]; a.axis('off')
    rows = [(r'$\sigma(t_x) = %.3f$' % (1/(1+np.exp(-t_x))), r'$b_x = \sigma(t_x) + c_x = %.3f$' % b_x),
            (r'$\sigma(t_y) = %.3f$' % (1/(1+np.exp(-t_y))), r'$b_y = \sigma(t_y) + c_y = %.3f$' % b_y),
            (r'$e^{t_w} = %.3f$' % np.exp(t_w), r'$b_w = p_w e^{t_w} = %.3f$' % b_w),
            (r'$e^{t_h} = %.3f$' % np.exp(t_h), r'$b_h = p_h e^{t_h} = %.3f$' % b_h)]
    for i_, (lhs, rhs) in enumerate(rows):
        a.text(0.02, 0.86 - i_*0.17, lhs, fontsize=12, color=INK2)
        a.text(0.40, 0.86 - i_*0.17, rhs, fontsize=12, color=C_RED)
    a.text(0.02, 0.14, 'centre offset is bounded in (0, 1):\nthe box centre CANNOT leave its cell.',
           fontsize=9.5, color=C_BLUE)
    a.set_xlim(0, 1); a.set_ylim(0, 1)
    plt.tight_layout(); plt.show()

interact(show_v2_decode,
         t_x=widgets.FloatSlider(value=0.0, min=-6, max=6, step=0.25, description='t_x', continuous_update=False),
         t_y=widgets.FloatSlider(value=0.0, min=-6, max=6, step=0.25, description='t_y', continuous_update=False),
         t_w=widgets.FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.1, description='t_w', continuous_update=False),
         t_h=widgets.FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.1, description='t_h', continuous_update=False),
         p_w=widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='prior p_w', continuous_update=False,
                                 style={'description_width': '80px'}),
         p_h=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.1, description='prior p_h', continuous_update=False,
                                 style={'description_width': '80px'}));

## 8.1 Where do the priors come from? k-means on the training boxes

v2's other idea: do not hand-pick the anchor shapes, **learn them from the data**. Run k-means over the ground truth box dimensions of the training set.

The twist is the distance metric. Plain Euclidean distance on $(w, h)$ would let large boxes dominate the objective simply because they have bigger numbers. The paper instead uses an **IoU-based distance**:

$$
d(\text{box}, \text{centroid}) = 1 - \text{IoU}(\text{box}, \text{centroid})
$$

which is exactly what we care about: how well does this prior *cover* that box, independent of scale. Below we run it on our own training boxes and measure the average IoU between each ground truth box and its assigned prior, as $k$ grows. This reproduces the trade-off curve in the paper: more priors means better coverage, at the cost of more predictions per cell.

In [ ]:
wh = np.array([[b[2] / IMG, b[3] / IMG] for boxes in BX_train for b in boxes])  # normalised w, h
print('collected %d ground truth boxes from the training set' % len(wh))

def kmeans_iou(wh, k, iters=40, seed=0):
    "k-means with d = 1 - IoU, comparing boxes by shape only (all centred at the origin)."
    r = np.random.default_rng(seed)
    cent = wh[r.choice(len(wh), k, replace=False)].copy()
    for _ in range(iters):
        inter = np.minimum(wh[:, None, 0], cent[None, :, 0]) * np.minimum(wh[:, None, 1], cent[None, :, 1])
        union = wh[:, None, 0]*wh[:, None, 1] + cent[None, :, 0]*cent[None, :, 1] - inter
        d = 1 - inter / union                       # the IoU distance from the paper
        assign = d.argmin(1)
        for i in range(k):
            if (assign == i).any():
                cent[i] = wh[assign == i].mean(0)
    return cent, (1 - d.min(1)).mean(), assign

ks = [1, 2, 3, 4, 5, 6, 7, 8]
mious = [kmeans_iou(wh, k)[1] for k in ks]
cent5, miou5, assign5 = kmeans_iou(wh, 5)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9))
ax[0].plot(ks, mious, marker='o', ms=6, color=C_BLUE)
ax[0].set_xlabel('k (number of anchor priors)'); ax[0].set_ylabel('mean IoU to assigned prior')
ax[0].set_title('More priors, better coverage, diminishing returns')
for k_, m_ in zip(ks, mious):
    if k_ in (1, 5, 8):
        ax[0].annotate('%.3f' % m_, (k_, m_), textcoords='offset points', xytext=(0, 9),
                       ha='center', fontsize=8.5, color=INK2)

ax[1].scatter(wh[:, 0], wh[:, 1], s=7, alpha=0.25, color=MUTED, label='ground truth boxes')
ax[1].scatter(cent5[:, 0], cent5[:, 1], s=140, color=C_RED, marker='X',
              edgecolor='white', linewidth=1.2, label='k=5 anchor priors', zorder=3)
ax[1].set_xlabel('width (fraction of image)'); ax[1].set_ylabel('height (fraction of image)')
ax[1].set_title('Priors land in the middle of the data'); ax[1].legend(fontsize=8.5)
plt.tight_layout(); plt.show()

print('k=5 anchor priors (w, h as fraction of image):')
for c_ in cent5[np.argsort(cent5[:, 0])]:
    print('   %.3f x %.3f   (%.1f x %.1f pixels)' % (c_[0], c_[1], c_[0]*IMG, c_[1]*IMG))
print('mean IoU of a ground truth box to its assigned prior: %.3f' % miou5)
print('\nNote: our shapes were sampled from a uniform blob, so the priors just tile that blob.')
print('On real data (VOC/COCO) k-means recovers meaningful modes: tall pedestrians, wide cars.')

---
# 9. YOLO v3: logistic classifiers and multi-scale prediction

**Bounding box prediction:**
- Uses **logistic regression** to predict an objectness score for each bounding box.
- Assigns only **one anchor box to each ground truth object**, the prior with the best IoU. Other anchors that overlap the object above a threshold are simply *ignored* (they contribute no loss at all), rather than being trained as negatives. This is a cleaner treatment of ambiguity than v1's "responsible box" election.

**Class prediction:**
- Utilises **binary cross-entropy for independent logistic classifiers**, one per class, instead of a softmax.
- This is a genuine modelling change, not a cosmetic one. Softmax *asserts* that classes are mutually exclusive: probabilities must sum to 1, so "Woman" and "Person" compete. Independent logistic classifiers allow **multiple labels for the same box**, which is what open datasets with overlapping labels (Open Images) actually need.
- Our v1 head uses softmax + SSE, faithful to v1. Swapping it to per-class BCE would be a two-line change.

**Multi-scale predictions:**
- Predicts boxes at **three different scales**, with **three anchor boxes at each scale**, to improve small object detection. v2 used five anchor boxes at one scale; v3 uses 3 x 3 = 9 priors spread across resolutions.
- Detecting small objects on a coarse $13 \times 13$ map is hopeless: the object is smaller than a cell. A finer $52 \times 52$ map has the resolution, but its features are shallow and semantically weak. This tension is exactly what **FPN** (Section 16) resolves, and v3 adopts an FPN-like top-down pathway to do it.

**New backbone:**
- **Darknet-53**: 53 convolutional layers with **residual connections**, a real improvement over v2's Darknet-19.
- **Spatial Pyramid Pooling (SPP)**: a modified SPP block concatenating multiple max-pooling outputs with different kernel sizes, which widens the receptive field cheaply without changing the spatial resolution.

---
# 10. YOLO architecture anatomy: Backbone, Neck, and Head

Every modern detector, YOLO or not, decomposes into three parts. This vocabulary is worth internalising because it is how the literature is organised:

- **Backbone**
  - Extracts features from the input image.
  - Typically a CNN trained on a large-scale image classification task (ImageNet), then transferred.
  - Examples: Darknet-19 (v2), Darknet-53 (v3), CSPDarknet53 (v4), ResNet (RetinaNet), VGG-16 (SSD).
- **Neck**
  - **Aggregates and refines features**, enhancing spatial and semantic information **across scales**.
  - This is where FPN, PANet, and SPP live. The neck is the part that fixes "high resolution but semantically weak".
- **Head**
  - Performs the final predictions (classification, localisation).
  - Includes post-processing such as **non-maximum suppression**.

Mapped onto the model we built in Section 6.3:

| Part | In our TinyYOLO | In YOLO v4 |
|---|---|---|
| Backbone | `self.backbone`, 4 conv/pool stages, 96 to 6 | CSPDarknet53 |
| Neck | *none* (we predict at a single scale) | SPP + PANet |
| Head | `self.head`, 1x1 conv to $B \cdot 5 + C$, then threshold + NMS | YOLO head at 3 scales |

Our model has **no neck at all**, which is precisely why it would fail on multi-scale objects. Sections 13 and 16 build the missing piece.

---
# 11. YOLO v4

**Enhanced architecture:**
- Utilises **CSPDarknet53** with **cross-stage partial connections** (CSPNet) for improved computational efficiency. CSP splits the feature map, sends one part through the dense block and the other straight to the end, then merges. This cuts duplicate gradient information and shrinks compute for the same accuracy.
- Employs the **Mish** activation function:

$$
f(x) = x \tanh(\text{softplus}(x)) = x \tanh\!\left(\ln(1 + e^{x})\right)
$$

**Bag-of-Specials (BoS)**, methods that add a small inference cost but noticeably improve accuracy:
- **SPP** (spatial pyramid pooling) to widen the receptive field.
- **PANet** (path aggregation network) to combine features across scales. FPN adds a top-down path; PANet adds a *bottom-up* path on top of it, so fine spatial detail can also travel back up.
- A modified **SAM** (spatial attention module) for better feature refinement.

**Bag-of-Freebies (BoF)**, methods that cost nothing at inference because they only change training:
- **Mosaic augmentation**: combines four images into one, improving the model's ability to detect objects **outside their usual context** and giving a big effective batch of varied scales for free.
- **DropBlock** regularisation replaces Dropout. Dropping isolated pixels of a conv feature map does little, since neighbours carry the same information; DropBlock drops contiguous *regions*, which actually removes semantic content.
- **Self-Adversarial Training (SAT)**: perturbs the input to deceive the model into thinking the ground truth object is not present, then trains on that image, improving robustness.
- **CIoU loss** for box regression, which accounts for overlap, centre distance, and aspect ratio consistency together, rather than regressing coordinates independently.
- **Hyperparameter optimisation** with genetic algorithms over the first 10% of training, plus cosine annealing of the learning rate.

Let us look at Mish next to the alternatives, since it is a concrete, checkable detail.

In [ ]:
x = np.linspace(-5, 3, 600)
mish = x * np.tanh(np.log1p(np.exp(x)))          # x * tanh(softplus(x))
relu = np.maximum(x, 0)
leaky = np.where(x > 0, x, 0.1 * x)
# numerical derivative of mish
dmish = np.gradient(mish, x)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(x, relu, color=MUTED, ls='--', label='ReLU')
ax[0].plot(x, leaky, color=C_YELLOW, ls=':', label='LeakyReLU(0.1)')
ax[0].plot(x, mish, color=C_BLUE, label='Mish')
ax[0].axhline(0, color=GRIDC, lw=0.8); ax[0].axvline(0, color=GRIDC, lw=0.8)
ax[0].set_title('Mish:  $f(x) = x \\tanh(\\mathrm{softplus}(x))$'); ax[0].legend(fontsize=8.5)
ax[0].set_xlabel('x'); ax[0].set_ylabel('f(x)')
ax[0].annotate('small negative values\nsurvive (no hard zero)', xy=(-1.5, mish[np.argmin(np.abs(x+1.5))]),
               xytext=(-4.8, 1.4), fontsize=8, color=C_BLUE,
               arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.0))

ax[1].plot(x, np.where(x > 0, 1.0, 0.0), color=MUTED, ls='--', label="ReLU'")
ax[1].plot(x, dmish, color=C_BLUE, label="Mish'")
ax[1].axhline(0, color=GRIDC, lw=0.8); ax[1].axvline(0, color=GRIDC, lw=0.8)
ax[1].set_title('Derivative: Mish is smooth everywhere, ReLU is not at 0')
ax[1].set_xlabel('x'); ax[1].legend(fontsize=8.5)
plt.tight_layout(); plt.show()

def mish_fn(v):
    return v * np.tanh(np.log1p(np.exp(v)))

print('Mish is smooth, non-monotonic, and self-regularising:')
print('  min value %.4f at x = %.2f  (a small negative dip ReLU cannot represent)'
      % (mish.min(), x[mish.argmin()]))
print('  mish(0)  = %+.5f' % mish_fn(0.0))
print('  mish(-4) = %+.5f  (nearly off, but not exactly zero: gradient still flows)' % mish_fn(-4.0))
print('  relu(-4) = %+.5f  (hard zero: this unit is dead, no gradient)' % max(-4.0, 0.0))

---
# 12. YOLO versions: the timeline

Reproduced from the lecture slides, after Terven & Cordova-Esparza, *A Comprehensive Review of YOLO Architectures in Computer Vision: From YOLOv1 to YOLOv8 and YOLO-NAS*, arXiv 2024.

| YOLO Version | Year | Main Developments |
|---|---|---|
| **YOLOv1** | 2016 | Introduced as a single-stage detector; divides the image into a grid and predicts bounding boxes and class probabilities. |
| **YOLOv2** | 2017 | Added anchor boxes, improved training with high-resolution images, and introduced passthrough layers for small object detection. Introduced multiscale training. |
| **YOLOv3** | 2018 | Predicts classes for each bounding box instead of grids, uses multi-scale predictions at three different scales for improved accuracy. |
| **YOLOv4** | 2020 | Enhanced using techniques like CSPNet, PANet, and mish activation; focused on improving both speed and accuracy. |
| **YOLOv5** | 2020 | No paper was published; included improvements like auto-anchor and scaled YOLO versions. |
| **YOLOv6** | 2022 | Introduced RepVGG-based backbone for better performance and used VariFocal loss for classification and SIoU/GIoU for regression. |
| **YOLOv7** | 2022 | Introduced Extended Efficient Layer Aggregation Network and Model Scaling for concatenation-based models. In the bag-of-freebies, included planned re-parameterized convolution and coarse label assignment for auxiliary head and fine label assignment for the lead head. |
| **YOLOv8** | 2023 | Used Anchor-free Split Ultralytics Head and supports variety of pretrained models. |
| **YOLO-NAS** | 2023 | Algorithm-generated model with quantization blocks that made the model faster and more accurate. |
| **YOLOv9** | 2024 | Introduced Information Bottleneck Principle, Reversible Functions, Programmable Gradient Information, Generalized Efficient Layer Aggregation Network. |
| **YOLOv10** | 2024 | Introduced consistent dual label assignments for NMS-free training, a holistic efficiency-accuracy driven model design, which outperformed previous models with lower latency and fewer parameters. |

Two arcs are worth noticing across this table:

- **Anchors appear and then disappear.** v1 was anchor-free, v2 to v7 are anchor-based, and v8 returns to **anchor-free**. Anchors were a crutch that made box regression learnable in 2017; better losses and label assignment eventually made them unnecessary.
- **NMS is the last hand-written component to fall.** v10 removes it with dual label assignments, making the detector genuinely **end-to-end**. Every version until then still ends with the greedy loop we wrote in Section 6.6.

---
# 13. SSD: Single Shot MultiBox Detector

> Liu et al, *SSD: Single Shot MultiBox Detector*, ECCV 2016

SSD is the other major single-stage detector of 2016, contemporary with YOLO. Its base network is VGG-16 through the Conv5_3 layer, with FC6 and FC7 converted to fully convolutional layers, followed by **extra feature layers** that keep shrinking.

**Multi-scale feature maps for detection:**
- Detection is performed on **several feature maps of decreasing resolution**, not just the last one.
- As feature map size decreases, the **scale of detected objects increases**. A $38 \times 38$ map detects small objects, a $1 \times 1$ map detects objects filling the image. This directly attacks YOLO v1's worst weakness.

**Exclusive convolutional predictors for each feature map:**
- Each detection map gets its **own** convolutional filters (a $3 \times 3$ conv), not shared with the others. Each predictor is free to specialise for its own scale.
- A feature layer of size $m \times n$ with $c$ channels gives $m \times n$ locations (grid cells). Bounding box offsets are predicted **relative to the default box location**, as in Faster R-CNN.

**Default boxes and aspect ratios:**
- For each of $k$ default boxes with different aspect ratios, SSD predicts $c$ class-specific scores and $4$ box offsets.
- For an $m \times n$ feature map there are therefore

$$
(c + 4)\,k\,m\,n \quad \text{outputs}
$$

Note the contrast with YOLO v1: here **every default box gets its own class scores** ($(c+4)k$ per location), whereas YOLO v1 shares one set of $C$ class probabilities across all $B$ boxes in a cell ($B \cdot 5 + C$). SSD can emit two different classes at one location; YOLO v1 cannot.

In [ ]:
def ssd_default_boxes(m, n, scale, aspect_ratios, extra_scale=None):
    '''SSD default boxes for an m x n feature map. Returns (cx, cy, w, h) in [0,1] image units.'''
    boxes = []
    for i in range(m):
        for j in range(n):
            cx, cy = (j + 0.5) / n, (i + 0.5) / m       # centre of each feature map cell
            for a in aspect_ratios:
                boxes.append((cx, cy, scale * math.sqrt(a), scale / math.sqrt(a)))
            if extra_scale is not None:                 # the extra a=1 box at scale sqrt(s_k s_k+1)
                boxes.append((cx, cy, extra_scale, extra_scale))
    return np.array(boxes)

# Reproduce the paper's figure: the SAME object seen by a coarse and a fine feature map.
img_demo = X_test[0]
gt_demo = BX_test[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(img_demo); draw_boxes(axes[0], gt_demo, lw=2.0)
axes[0].set_title('(a) image with GT boxes', fontsize=10)

for ax, (m, scale, ars, name) in zip(axes[1:], [
        (8, 0.20, [1.0, 2.0, 0.5], '(b) 8 x 8 feature map: small default boxes'),
        (4, 0.45, [1.0, 2.0, 0.5], '(c) 4 x 4 feature map: large default boxes')]):
    ax.imshow(img_demo, alpha=0.55)
    draw_grid(ax, m, color='white', lw=0.7, alpha=0.6)
    ci = m // 2
    # every location on this feature map carries a default box (faint dots),
    # and we draw the full set of k aspect ratios at one central location
    for (cx, cy, w, h) in ssd_default_boxes(m, m, scale, ars):
        ax.plot([cx*IMG], [cy*IMG], marker='.', ms=2, color='white', alpha=0.5)
    cx0, cy0 = (ci + 0.5)/m*IMG, (ci + 0.5)/m*IMG
    for a_, col in zip(ars, [C_BLUE, C_AQUA, C_YELLOW]):
        w_, h_ = scale*math.sqrt(a_)*IMG, scale/math.sqrt(a_)*IMG
        ax.add_patch(patches.Rectangle((cx0-w_/2, cy0-h_/2), w_, h_, fill=False,
                                       edgecolor=col, lw=1.8, ls=(0, (4, 2))))
    ax.plot([cx0], [cy0], marker='o', ms=5, color='white', mec=INK, mew=1.0)
    ax.set_title(name, fontsize=10)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_xlim(0, IMG); ax.set_ylim(IMG, 0)
plt.suptitle('SSD: the same location, tiled with default boxes of k aspect ratios, at every scale', y=1.0)
plt.tight_layout(); plt.show()

# The real SSD300 configuration reproduces the paper's famous 8732 boxes.
ssd300 = [('conv4_3', 38, 4), ('conv7 (fc7)', 19, 6), ('conv8_2', 10, 6),
          ('conv9_2', 5, 6), ('conv10_2', 3, 4), ('conv11_2', 1, 4)]
print('SSD300 default boxes per feature map:')
total = 0
for name, m, k in ssd300:
    n_ = m * m * k; total += n_
    print('   %-12s %2d x %2d x %d aspect ratios = %5d boxes' % (name, m, m, k, n_))
print('   %-12s %27s %5d boxes  <- the paper reports 8732' % ('TOTAL', '', total))
assert total == 8732

### Slider: feature map resolution, aspect ratios, and the $(c+4)kmn$ output count

Watch two things as you move the sliders:
- A finer feature map ($m$ large) produces **many small** default boxes; a coarse one produces **few large** ones. This is the multi-scale principle in one picture.
- The output count $(c+4)kmn$ grows **quadratically in $m$**. This is why SSD ends up with 8732 boxes and RetinaNet with ~100k: dense multi-scale prediction generates an enormous number of candidates, almost all of them background. Section 17 is the bill for this.

In [ ]:
def show_ssd(m=8, scale=0.20, n_ratios=3, c_classes=21):
    ar_pool = [1.0, 2.0, 0.5, 3.0, 1.0/3.0]
    ars = ar_pool[:n_ratios]
    k = len(ars)
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3), gridspec_kw={'width_ratios': [1, 1]})

    ax[0].imshow(img_demo, alpha=0.5)
    draw_grid(ax[0], m, color='white', lw=0.6, alpha=0.55)
    for i in range(m):
        for j in range(m):
            ax[0].plot([(j+0.5)/m*IMG], [(i+0.5)/m*IMG], marker='.', ms=2, color='white', alpha=0.45)
    cx0 = cy0 = (m//2 + 0.5)/m*IMG
    for a_, col in zip(ars, PALETTE):
        w_, h_ = scale*math.sqrt(a_)*IMG, scale/math.sqrt(a_)*IMG
        ax[0].add_patch(patches.Rectangle((cx0-w_/2, cy0-h_/2), w_, h_, fill=False,
                                          edgecolor=col, lw=1.8, ls=(0, (4, 2)),
                                          label='ratio %.2f' % a_))
    ax[0].plot([cx0], [cy0], marker='o', ms=5, color='white', mec=INK, mew=1.0)
    ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].grid(False)
    ax[0].set_xlim(0, IMG); ax[0].set_ylim(IMG, 0)
    ax[0].legend(fontsize=7.5, loc='upper right')
    ax[0].set_title('%d x %d feature map, k = %d default boxes per location' % (m, m, k), fontsize=10)

    ax[1].axis('off')
    n_out = (c_classes + 4) * k * m * m
    n_boxes = k * m * m
    ax[1].text(0.0, 0.86, r'$(c + 4)\,k\,m\,n$', fontsize=17, color=INK)
    ax[1].text(0.0, 0.68, '= (%d + 4) x %d x %d x %d' % (c_classes, k, m, m), fontsize=13, color=INK2)
    ax[1].text(0.0, 0.52, '= %s outputs' % format(n_out, ','), fontsize=16, color=C_BLUE, weight='bold')
    ax[1].text(0.0, 0.34, '= %s default boxes on this map' % format(n_boxes, ','), fontsize=12, color=INK2)
    ax[1].text(0.0, 0.16, 'each box: %d class scores + 4 offsets' % c_classes, fontsize=10, color=MUTED)
    ax[1].text(0.0, 0.02, 'of these, typically fewer than 10 are positive.', fontsize=10, color=C_RED)
    ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

interact(show_ssd,
         m=widgets.IntSlider(value=8, min=1, max=19, step=1, description='map size m', continuous_update=False,
                             style={'description_width': '90px'}),
         scale=widgets.FloatSlider(value=0.20, min=0.05, max=0.9, step=0.05, description='scale s_k',
                                   continuous_update=False, style={'description_width': '90px'}),
         n_ratios=widgets.IntSlider(value=3, min=1, max=5, step=1, description='k (ratios)',
                                    continuous_update=False, style={'description_width': '90px'}),
         c_classes=widgets.IntSlider(value=21, min=2, max=81, step=1, description='c (classes)',
                                     continuous_update=False, style={'description_width': '90px'}));

---
# 14. SSD: the loss function

The overall objective is a **weighted sum of the localization loss (loc) and the confidence loss (conf)**:

$$
L(x, c, l, g) = \frac{1}{N}\left( L_{conf}(x, c) + \alpha L_{loc}(x, l, g) \right)
$$

where:
- $x^p_{ij} \in \{1, 0\}$ indicates whether the $i^{\text{th}}$ default box **matches** the $j^{\text{th}}$ ground truth box of category $p$.
- $c$ = class probabilities, $l$ = predicted box parameters, $g$ = ground truth box parameters.
- $N$ is the number of matched default boxes. **If $N = 0$ the loss is set to 0**, otherwise dividing by zero on an image with no matches would destroy training.

**Localization loss** is a smooth L1 (Section 1) between the predicted box $l$ and the ground truth $\hat{g}$, regressing **offsets relative to the default box $d$**:

$$
L_{loc}(x, l, g) = \sum_{i \in Pos}^{N} \sum_{m \in \{cx, cy, w, h\}} x^k_{ij} \; \text{smooth}_{L1}\left(l^m_i - \hat{g}^m_j\right)
$$

$$
\hat{g}^{cx}_j = \frac{g^{cx}_j - d^{cx}_i}{d^w_i}
\qquad
\hat{g}^{cy}_j = \frac{g^{cy}_j - d^{cy}_i}{d^h_i}
\qquad
\hat{g}^{w}_j = \log\!\left(\frac{g^w_j}{d^w_i}\right)
\qquad
\hat{g}^{h}_j = \log\!\left(\frac{g^h_j}{d^h_i}\right)
$$

Look closely at what this encoding does, because it is the same idea as YOLO v2's decode read backwards:
- The **centre** offset is divided by the default box's own size, making it **scale invariant**: "a quarter of a box-width to the right" means the same thing for a big box and a small one.
- The **size** is regressed in **log space**, so the target is 0 when the box is exactly the right size, and doubling versus halving are symmetric.
- Compare with $b_w = p_w e^{t_w}$ from Section 8: identical relationship, just solved for $t_w$.

**Confidence loss** is the softmax loss over multiple class confidences:

$$
L_{conf}(x, c) = -\sum_{i \in Pos}^{N} x^p_{ij} \log(\hat{c}^p_i) \;-\; \sum_{i \in Neg} \log(\hat{c}^0_i)
\qquad \text{where} \qquad
\hat{c}^p_i = \frac{\exp(c^p_i)}{\sum_p \exp(c^p_i)}
$$

The second sum, over **negatives** pushing up the background class $\hat{c}^0$, is the term that will explode in Section 19. Let us verify the offset encoding is a lossless round trip, exactly as we did for YOLO.

In [ ]:
def ssd_encode(g, d):
    "Ground truth g and default box d, both (cx, cy, w, h). Returns the regression target g_hat."
    return np.array([(g[0] - d[0]) / d[2],          # centre offset, scaled by the default box width
                     (g[1] - d[1]) / d[3],
                     np.log(g[2] / d[2]),           # size in log space
                     np.log(g[3] / d[3])])

def ssd_decode(l, d):
    "Invert ssd_encode: predicted offsets l + default box d -> an absolute box."
    return np.array([l[0] * d[2] + d[0],
                     l[1] * d[3] + d[1],
                     d[2] * np.exp(l[2]),
                     d[3] * np.exp(l[3])])

d_box = np.array([0.50, 0.50, 0.20, 0.30])       # a default box
g_box = np.array([0.56, 0.47, 0.26, 0.24])       # a ground truth box it matched
g_hat = ssd_encode(g_box, d_box)
back = ssd_decode(g_hat, d_box)
print('default box d      :', np.round(d_box, 4))
print('ground truth g     :', np.round(g_box, 4))
print('encoded target ghat:', np.round(g_hat, 4), ' <- what the network regresses with smooth L1')
print('decoded back       :', np.round(back, 4))
assert np.allclose(back, g_box, atol=1e-9)
print('\nPASS: SSD offset encode -> decode is an exact round trip.')

print('\nA perfect prediction (l = ghat) has zero loss. Two ways to be wrong:')
for name, l_pred in [('predicts the default box itself (l=0)', np.zeros(4)),
                     ('half the correct width', ssd_encode(g_box*np.array([1,1,0.5,1]), d_box))]:
    resid = l_pred - g_hat
    sl1 = np.where(np.abs(resid) < 1.0, 0.5*resid**2, np.abs(resid) - 0.5).sum()
    print('   %-38s smooth L1 loss = %.4f' % (name, sl1))

---
# 15. SSD: practical implementation

**Hard negative mining:**
- Similar to Faster R-CNN, **most anchor boxes are negative**. After matching, the vast majority of the 8732 default boxes have no ground truth.
- To counter it, SSD does not use all negatives. It **sorts them by their confidence loss (highest first)** and keeps only the worst offenders, so that the ratio between negatives and positives is at most $\sim 3 : 1$.
- The intuition: a negative the model already classifies confidently as background teaches it nothing. The negatives worth training on are the ones it is currently getting **wrong**.
- This works, and it is what makes SSD trainable at all. But note what it is: a **hard, discrete, hand-tuned rule** that throws data away. Section 18 explains why it only partly fixes the problem, and Section 20 gives the smooth version.

**Data augmentation.** Each training sample is obtained by one of:
- Using the original image.
- Sampling a patch such that the minimum IoU with the objects is in $\{0.1, 0.3, 0.5, 0.7, 0.9\}$.
- Randomly sampling a patch.

This is more important than it looks: random crops **synthesise scale variation**, which is exactly what a multi-scale detector needs to learn its per-scale predictors.

Let us implement hard negative mining and watch what it selects.

In [ ]:
rng_hn = np.random.default_rng(4)
N_DEF = 8732                                   # SSD300's default box count
n_pos = 12
# Simulate a partly-trained model's background-class confidence for each default box.
conf_bg = np.concatenate([rng_hn.beta(12, 1.2, N_DEF - n_pos),   # negatives: mostly confident bg
                          rng_hn.beta(2, 4, n_pos)])             # positives: low bg confidence
is_pos_hn = np.zeros(N_DEF, bool); is_pos_hn[-n_pos:] = True
conf_loss = -np.log(np.clip(np.where(is_pos_hn, 1 - conf_bg, conf_bg), 1e-9, 1))

neg_idx = np.where(~is_pos_hn)[0]
order = neg_idx[np.argsort(-conf_loss[neg_idx])]      # sort negatives by confidence loss, desc
n_keep = 3 * n_pos                                    # the 3:1 rule
kept = order[:n_keep]

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.7))
ax[0].hist(conf_loss[~is_pos_hn], bins=60, color=MUTED, alpha=0.75, label='all %d negatives' % (N_DEF-n_pos))
ax[0].hist(conf_loss[kept], bins=60, color=C_RED, label='%d kept (hardest)' % n_keep)
ax[0].axvline(conf_loss[kept].min(), color=C_RED, ls='--', lw=1.2)
ax[0].set_yscale('log'); ax[0].set_xlabel('confidence loss of a negative default box')
ax[0].set_ylabel('count (log)'); ax[0].set_title('Hard negative mining keeps only the right tail')
ax[0].legend(fontsize=8.5)

tot_all = conf_loss[~is_pos_hn].sum()
tot_kept = conf_loss[kept].sum()
tot_pos = conf_loss[is_pos_hn].sum()
bars = ax[1].bar(['negatives\n(all %d)' % (N_DEF-n_pos), 'negatives\n(kept %d)' % n_keep,
                  'positives\n(%d)' % n_pos], [tot_all, tot_kept, tot_pos],
                 color=[MUTED, C_RED, C_AQUA], width=0.6)
for b_, v_ in zip(bars, [tot_all, tot_kept, tot_pos]):
    ax[1].text(b_.get_x()+b_.get_width()/2, v_, ' %.1f' % v_, ha='center', va='bottom', fontsize=9, color=INK2)
ax[1].set_ylabel('summed confidence loss'); ax[1].set_title('Total loss contributed')
plt.tight_layout(); plt.show()

print('Without mining: negatives contribute %.1f vs positives %.1f  ->  %.0fx more'
      % (tot_all, tot_pos, tot_all/tot_pos))
print('With 3:1 mining: negatives contribute %.1f vs positives %.1f  ->  %.1fx more'
      % (tot_kept, tot_pos, tot_kept/tot_pos))
print('\nMining fixes the ratio, but note what it did: it DISCARDED %s negatives outright.'
      % format(N_DEF - n_pos - n_keep, ','))
print('It is a hard cutoff. Focal loss (Section 20) keeps every example and re-weights smoothly.')

---
# 16. Feature Pyramid Network

> Lin et al, *Feature Pyramid Networks for Object Detection*, CVPR 2017

The problem, stated precisely:
- Feature maps from initial layers are **high resolution but not semantically strong**, so they cannot be used for detection on their own. They have seen only a few convolutions: they know about edges, not about objects.
- Feature maps from deep layers are **semantically rich but low resolution**, so small objects have been downsampled away.
- SSD's answer was to predict from each level independently, which means its high-resolution predictions are made from semantically weak features.
- **FPN provides a top-down pathway to construct higher resolution layers from a semantically rich layer**, so that *every* level of the pyramid is both high resolution and semantically strong.

**Methodology:**
- All convolutional feature maps $C_i$ are treated with a **$1 \times 1$ convolution with 256 channels** (the *lateral* connection), which puts every level in a common channel space so they can be added.
- $M_5$ is **upsampled by a factor of 2**.
- $M_5$ and $C_4$ signals are **element-wise added** to give $M_4$; similarly followed in the downward direction.
- A **$3 \times 3$ convolution** is used on each $M_i$ to reduce the **aliasing effect** of upsampling, producing $P_i$.
- Finally, the $P_i$ are individually fed into exclusive object detectors.

Note the design economy: the top-down path itself is only **1x1 convs, nearest-neighbour upsampling, and addition**. There is no fancy machinery here at all, yet it is one of the highest-value ideas in detection. The bulk of FPN's parameters are actually the three $3 \times 3$ smoothing convs at 256 channels; on a real ResNet-50 backbone the whole pyramid adds roughly 13% more parameters, which is a modest price for making *every* level semantically strong. (On the deliberately tiny toy backbone below it will look far more expensive, for the obvious reason.) Let us build it and print real shapes.

In [ ]:
class ToyBackbone(nn.Module):
    "A stand-in for ResNet: five stride-2 stages. We tap the last three (C3, C4, C5)."
    def __init__(self):
        super().__init__()
        def blk(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=2, padding=1),
                                 nn.BatchNorm2d(cout), nn.ReLU())
        self.s1, self.s2 = blk(3, 16), blk(16, 32)
        self.s3, self.s4, self.s5 = blk(32, 64), blk(64, 128), blk(128, 256)
    def forward(self, x):
        x = self.s2(self.s1(x))
        c3 = self.s3(x); c4 = self.s4(c3); c5 = self.s5(c4)
        return c3, c4, c5

class FPN(nn.Module):
    def __init__(self, in_channels=(64, 128, 256), out_channels=256):
        super().__init__()
        # lateral 1x1 convs: put every level into the same 256-channel space so they can be added
        self.lateral = nn.ModuleList([nn.Conv2d(c, out_channels, 1) for c in in_channels])
        # 3x3 convs on each merged map, to reduce the aliasing introduced by upsampling
        self.smooth = nn.ModuleList([nn.Conv2d(out_channels, out_channels, 3, padding=1)
                                     for _ in in_channels])
    def forward(self, c3, c4, c5):
        m5 = self.lateral[2](c5)                                        # top of the pyramid
        m4 = self.lateral[1](c4) + F.interpolate(m5, size=c4.shape[-2:], mode='nearest')  # 2x up, add
        m3 = self.lateral[0](c3) + F.interpolate(m4, size=c3.shape[-2:], mode='nearest')  # 2x up, add
        p3, p4, p5 = self.smooth[0](m3), self.smooth[1](m4), self.smooth[2](m5)
        return (p3, p4, p5), (m3, m4, m5)

torch.manual_seed(SEED)
backbone_fpn = ToyBackbone().to(device)
fpn = FPN().to(device)

x_fpn = torch.randn(1, 3, 256, 256, device=device)
with torch.no_grad():
    c3, c4, c5 = backbone_fpn(x_fpn)
    (p3, p4, p5), (m3, m4, m5) = fpn(c3, c4, c5)

print('input: %s\n' % (tuple(x_fpn.shape),))
print('%-6s %-22s %-8s %s' % ('', 'shape', 'stride', 'role'))
print('-' * 74)
for nm, t, role in [('C3', c3, 'bottom-up: high res, semantically WEAK'),
                    ('C4', c4, 'bottom-up: medium'),
                    ('C5', c5, 'bottom-up: low res, semantically STRONG')]:
    print('%-6s %-22s %-8s %s' % (nm, tuple(t.shape), '/%d' % (x_fpn.shape[-1]//t.shape[-1]), role))
print()
for nm, t, role in [('P3', p3, 'top-down: high res AND semantically strong'),
                    ('P4', p4, 'top-down: medium res AND semantically strong'),
                    ('P5', p5, 'top-down: low res, semantically strong')]:
    print('%-6s %-22s %-8s %s' % (nm, tuple(t.shape), '/%d' % (x_fpn.shape[-1]//t.shape[-1]), role))

print('\nEvery P has exactly 256 channels: %s' % ([p.shape[1] for p in (p3, p4, p5)]))
print('P_i keeps the resolution of C_i: %s vs %s'
      % ([tuple(p.shape[-2:]) for p in (p3, p4, p5)], [tuple(c.shape[-2:]) for c in (c3, c4, c5)]))
n_bb = sum(p.numel() for p in backbone_fpn.parameters())
n_fpn = sum(p.numel() for p in fpn.parameters())
n_lat = sum(p.numel() for p in fpn.lateral.parameters())
n_sm = sum(p.numel() for p in fpn.smooth.parameters())
print('\nbackbone params %s | FPN params %s' % (format(n_bb, ','), format(n_fpn, ',')))
print('   of which lateral 1x1 convs: %s (the actual top-down pathway is nearly free)'
      % format(n_lat, ','))
print('   of which 3x3 smooth convs : %s (this is where FPN spends its parameters)'
      % format(n_sm, ','))
print('Our toy backbone is deliberately tiny, so FPN looks huge next to it (%.0f%% overhead).'
      % (100*n_fpn/n_bb))
print('On a real ResNet-50 (~25.6M params) the same FPN adds ~3.3M, about 13%: a modest price')
print('for making every pyramid level both high resolution AND semantically strong.')

Now the picture: the bottom-up pathway on the left, the top-down pathway on the right, with real tensor shapes. Box area is drawn proportional to the feature map's spatial resolution, and colour saturation stands for semantic strength.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.4))
levels = [('C3', c3, 'P3', p3), ('C4', c4, 'P4', p4), ('C5', c5, 'P5', p5)]
ys = [0.5, 2.0, 3.5]           # C3 at the bottom (high res), C5 at the top (low res)

for (cn, ct, pn, pt), y in zip(levels, ys):
    res = ct.shape[-1]
    w_ = 0.9 * res / 32.0                                  # width proportional to resolution
    sem = {'C3': 0.12, 'C4': 0.36, 'C5': 0.85}[cn]         # semantic strength, faked for the picture
    ax.add_patch(patches.Rectangle((1.6 - w_/2, y - 0.22), w_, 0.44,
                                   facecolor=C_BLUE, alpha=sem, edgecolor=C_BLUE, lw=1.4))
    ax.text(1.6, y + 0.36, '%s  %dx%d x%d' % (cn, ct.shape[-2], ct.shape[-1], ct.shape[1]),
            ha='center', fontsize=8.5, color=INK2)
    ax.add_patch(patches.Rectangle((5.4 - w_/2, y - 0.22), w_, 0.44,
                                   facecolor=C_AQUA, alpha=0.85, edgecolor=C_AQUA, lw=1.4))
    ax.text(5.4, y + 0.36, '%s  %dx%d x%d' % (pn, pt.shape[-2], pt.shape[-1], pt.shape[1]),
            ha='center', fontsize=8.5, color=INK2)
    ax.annotate('', xy=(4.75, y), xytext=(2.45, y),
                arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.3))
    ax.text(3.6, y + 0.06, '1x1 conv, 256ch', ha='center', fontsize=7.5, color=MUTED)
    ax.annotate('', xy=(6.85, y), xytext=(6.05, y),
                arrowprops=dict(arrowstyle='->', color=C_AQUA, lw=1.6))
    ax.text(7.6, y, 'predict', ha='center', va='center', fontsize=8.5, color=C_AQUA, weight='bold')

for y_hi, y_lo in [(3.5, 2.0), (2.0, 0.5)]:
    ax.annotate('', xy=(5.4, y_lo + 0.26), xytext=(5.4, y_hi - 0.26),
                arrowprops=dict(arrowstyle='->', color=C_RED, lw=2.0))
    ax.text(5.62, (y_hi + y_lo)/2, 'upsample 2x\nthen add', fontsize=7.5, color=C_RED, va='center')

for x_, t_ in [(1.6, 'bottom-up\n(backbone)'), (5.4, 'top-down\n(FPN)')]:
    ax.text(x_, 4.35, t_, ha='center', fontsize=11, color=INK, weight='bold')
ax.text(1.6, -0.35, 'fading colour = weak semantics\nat high resolution', ha='center',
        fontsize=8, color=MUTED)
ax.text(5.4, -0.35, 'every level is now\nsemantically strong', ha='center', fontsize=8, color=C_AQUA)
ax.annotate('', xy=(0.35, 0.3), xytext=(0.35, 3.7), arrowprops=dict(arrowstyle='->', color=INK2, lw=1.2))
ax.text(0.18, 2.0, 'resolution increases', rotation=90, va='center', fontsize=8.5, color=INK2)
ax.set_xlim(0, 8.6); ax.set_ylim(-0.75, 4.75); ax.axis('off')
ax.set_title('FPN: the top-down pathway carries semantics back down to high resolution', fontsize=11)
plt.tight_layout(); plt.show()

---
# 17. RetinaNet: why are two-stage detectors more accurate?

> Lin et al, *Focal Loss for Dense Object Detection*, ICCV 2017

**Intuition:** two-stage detectors are more accurate than one-stage detectors due to **lesser class imbalance** between background (negative) and object-containing (positive) proposals.

- **Two-stage detectors**: Selective Search or an RPN narrows the field to **1k to 2k regions**, and those are already filtered to be object-like. A *lesser* background-to-object ratio reaches the classifier. The proposal stage is, among other things, a **class-balancing mechanism**, and nobody had quite named it as such before this paper.
- **One-stage detectors**: dense sampling over a sliding window yields **~100k regions** per image, of which maybe a dozen contain objects. A *very high* background-to-object ratio.

So the accuracy gap was never really about the architecture. It was about the **loss being computed over a hopelessly imbalanced sample**. That reframing is the paper's contribution; focal loss is just the fix that follows from it.

---
# 18. Class imbalance in object detection: the problems

- **Training is inefficient as easy negatives contribute no useful signal.** A patch of empty sky that the model already scores as background at 99% teaches it nothing. The gradient is real but it points nowhere new.
- **Loss due to easy negatives overwhelms loss due to positives**, and thereby the training process **can lead to degenerate models**. Each easy negative contributes a tiny loss, but there are 100,000 of them and only ~20 positives. Tiny multiplied by enormous beats large multiplied by few.
- **Hard negative training alleviates it to some extent** (this is SSD's 3:1 mining from Section 15), but only partly: it is a hard threshold, it discards most of the data, and the ratio it enforces is another hand-tuned constant.

We saw a small version of this ourselves in Section 6.2: `noobj` was the first term to collapse, and $\lambda_{\text{noobj}} = 0.5$ exists solely to keep it from swamping everything else.

---
# 19. Cross entropy is the wrong loss here

CE loss for binary classification is typically implemented as:

$$
\text{CE}(p, y) =
\begin{cases}
-\log(p) & \text{if } y = 1 \\
-\log(1 - p) & \text{otherwise}
\end{cases}
$$

which is rewritten compactly by defining $p_t$ as the probability **of the ground truth class**:

$$
p_t =
\begin{cases}
p & \text{if } y = 1 \\
1 - p & \text{otherwise}
\end{cases}
\qquad \Longrightarrow \qquad
\text{CE}(p, y) = \text{CE}(p_t) = -\log(p_t)
$$

The critical observation, visible in the $\gamma = 0$ curve we plot next:

> **Easily classified examples ($p_t \gg 0.5$) incur a loss of non-trivial magnitude.**

Concretely: an anchor classified correctly with $p_t = 0.9$ still contributes $-\log(0.9) = 0.105$. That looks negligible. Multiply by 100,000 anchors and it is **10,500**, against maybe 20 positives contributing ~14 in total. The background wins by three orders of magnitude, and the model's best move is to predict "background" everywhere. That is the degenerate model.

---
# 20. Balanced cross entropy, and focal loss

**Balanced Cross Entropy:**
- Introduces a weighting factor $\alpha \in [0, 1]$, which can be the inverse class frequency or a hyperparameter.
- The $\alpha$-balanced CE loss is given by:

$$
\text{CE}(p_t) = -\alpha \log(p_t)
$$

- This addresses the **positive/negative** balance, but note what it does *not* do: it scales easy and hard examples by the same constant. It rebalances *classes*, not *difficulty*. A model still spends most of its capacity on examples it has already mastered.

**Focal Loss:**
- Adds a **modulating factor** $(1 - p_t)^\gamma$ to the CE loss, with a tunable **focusing parameter** $\gamma \ge 0$:

$$
\text{FL}(p_t) = -(1 - p_t)^{\gamma} \log(p_t)
$$

and in practice the paper uses the $\alpha$-balanced variant, combining both ideas:

$$
\text{FL}(p_t) = -\alpha_t (1 - p_t)^{\gamma} \log(p_t)
\qquad \text{with } \gamma = 2, \; \alpha = 0.25
$$

Read the modulating factor carefully:
- When an example is **misclassified** ($p_t$ small), $(1 - p_t)^\gamma \approx 1$: the loss is **unchanged**. Hard examples keep their full gradient.
- When an example is **easily classified** ($p_t \to 1$), $(1 - p_t)^\gamma \to 0$: the loss is **driven towards zero**. At $p_t = 0.9$ and $\gamma = 2$, the loss is scaled by $(0.1)^2 = 0.01$, a **100x** down-weighting.
- $\gamma = 0$ **recovers plain CE exactly**, so focal loss is a strict generalisation, not a different loss.
- $\gamma$ smoothly controls *how aggressively* easy examples are discounted. Unlike hard negative mining, **nothing is discarded**: every anchor still contributes, just proportionally to how much it still has to teach.

In [ ]:
def focal_loss(logits, targets, gamma=2.0, alpha=0.25, reduction='none'):
    '''Focal loss from scratch, on raw logits (numerically stable).
    gamma = 0 and alpha = 0.5 recovers (half) standard binary cross entropy.'''
    p = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')   # -log(p_t)
    p_t = p * targets + (1 - p) * (1 - targets)                                  # prob of TRUE class
    modulating = (1.0 - p_t) ** gamma                                            # the focal term
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)                      # the balancing term
    loss = alpha_t * modulating * ce
    return loss.mean() if reduction == 'mean' else (loss.sum() if reduction == 'sum' else loss)

# --- check 1: gamma=0, alpha=0.5 must be exactly 0.5 * BCE ---
_lg = torch.randn(1000); _tg = (torch.rand(1000) > 0.5).float()
_fl = focal_loss(_lg, _tg, gamma=0.0, alpha=0.5)
_bce = F.binary_cross_entropy_with_logits(_lg, _tg, reduction='none')
assert torch.allclose(_fl, 0.5 * _bce, atol=1e-6)
print('PASS: focal loss with gamma=0, alpha=0.5 reduces exactly to 0.5 * BCE (gamma=0 recovers CE).')

# --- check 2: the modulating factor down-weights by the amount we claim ---
print('\ndown-weighting applied by (1 - p_t)^gamma:')
print('%8s %12s %12s %12s' % ('p_t', 'gamma=0', 'gamma=2', 'factor'))
for p_t in [0.1, 0.5, 0.7, 0.9, 0.99]:
    ce_v = -math.log(p_t); fl_v = (1-p_t)**2 * ce_v
    print('%8.2f %12.4f %12.4f %11.0fx' % (p_t, ce_v, fl_v, ce_v/max(fl_v, 1e-12)))

### The focal loss curves (reproducing the paper's figure)

$\gamma$ is an **ordered** quantity, so the curves use a single-hue ramp: the darker the line, the stronger the focusing. Each curve is also labelled directly.

In [ ]:
p_t = np.linspace(0.001, 1.0, 800)
gammas = [0, 0.5, 1, 2, 5]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for g, col in zip(gammas, RAMP_ORDINAL):
    ax[0].plot(p_t, -(1 - p_t)**g * np.log(p_t), color=col,
               label=r'$\gamma = %s$' % g, lw=2.2 if g == 2 else 1.8)
ax[0].axvspan(0.6, 1.0, color=MUTED, alpha=0.08)
ax[0].text(0.80, 4.4, 'well-classified\nexamples', ha='center', fontsize=9, color=INK2)
ax[0].annotate('CE still charges 0.105 here', xy=(0.9, -np.log(0.9)), xytext=(0.42, 1.55),
               fontsize=8.5, color=RAMP_ORDINAL[0],
               arrowprops=dict(arrowstyle='->', color=RAMP_ORDINAL[0], lw=1.1))
ax[0].set_xlabel('probability of ground truth class,  $p_t$'); ax[0].set_ylabel('loss')
ax[0].set_ylim(0, 5); ax[0].set_xlim(0, 1)
ax[0].set_title(r'$\mathrm{CE}(p_t) = -\log(p_t)$   vs   $\mathrm{FL}(p_t) = -(1-p_t)^\gamma \log(p_t)$')
ax[0].legend(fontsize=9, title='focusing', title_fontsize=8)
for g, col in zip(gammas, RAMP_ORDINAL):
    y_ = -(1 - 0.62)**g * np.log(0.62)
    ax[0].text(0.635, y_ + 0.04, r'$\gamma=%s$' % g, fontsize=7.5, color=col)

# zoom on the region that actually matters: the easy examples
for g, col in zip(gammas, RAMP_ORDINAL):
    ax[1].plot(p_t, -(1 - p_t)**g * np.log(p_t), color=col, label=r'$\gamma = %s$' % g)
ax[1].set_xlim(0.5, 1.0); ax[1].set_ylim(0, 0.22)
ax[1].set_xlabel('probability of ground truth class,  $p_t$'); ax[1].set_ylabel('loss')
ax[1].set_title('Zoom on the easy examples (where the 100k anchors live)')
ax[1].legend(fontsize=8.5)
plt.tight_layout(); plt.show()

print('loss at p_t = 0.9 (a confidently correct background anchor):')
for g in gammas:
    print('   gamma = %-4s ->  %.5f   (x100,000 anchors = %9.1f)'
          % (g, (1-0.9)**g * -np.log(0.9), 1e5 * (1-0.9)**g * -np.log(0.9)))

That last printout is the entire argument for RetinaNet in five lines. With plain CE, one hundred thousand *confidently correct* background anchors contribute **10,536** to the loss. With $\gamma = 2$ they contribute **105**. The positives (~20 of them, contributing ~14) go from being invisible to being competitive.

### The experiment: 100,000 anchors, 20 positives

Let us make this fully concrete on a synthetic anchor set with a realistic imbalance, of the kind a partly-trained dense detector produces:

- **100,000 anchors**, of which **20 are positives** (roughly RetinaNet's real numbers for one image).
- Most negatives are **easy**: the model already assigns them a very low foreground probability.
- A small fraction (1%) are **hard** negatives: genuinely ambiguous patches near object boundaries.
- We define an **easy negative** as any negative with $p_t \ge 0.9$, and account for where the total loss comes from under CE versus focal loss.

In [ ]:
def make_anchor_set(n_anchors=100_000, n_pos=20, hard_frac=0.01, seed=0):
    '''Simulate one image's worth of anchor predictions from a partly-trained dense detector.
    Returns p_t (probability assigned to the TRUE class) and the positive mask.'''
    r = np.random.default_rng(seed)
    n_neg = n_anchors - n_pos
    n_hard = int(n_neg * hard_frac); n_easy = n_neg - n_hard
    p_fg_easy = r.beta(1.0, 60.0, n_easy)     # easy negatives: foreground prob near 0
    p_fg_hard = r.beta(5.0, 5.0, n_hard)      # hard negatives: genuinely ambiguous, around 0.5
    p_fg_pos = r.beta(5.0, 3.0, n_pos)        # positives: the model is doing okay but not perfect
    p_fg = np.concatenate([p_fg_easy, p_fg_hard, p_fg_pos])
    is_pos = np.zeros(n_anchors, bool); is_pos[-n_pos:] = True
    p_t = np.where(is_pos, p_fg, 1 - p_fg)    # probability of the TRUE class
    return np.clip(p_t, 1e-8, 1 - 1e-8), is_pos

def loss_breakdown(p_t, is_pos, gamma, alpha=None):
    '''Total loss and the share contributed by easy negatives / hard negatives / positives.
    alpha=None means no alpha weighting at all (a_t = 1), which with gamma=0 is plain CE.'''
    a_t = 1.0 if alpha is None else np.where(is_pos, alpha, 1 - alpha)
    L = -a_t * (1 - p_t)**gamma * np.log(p_t)
    easy = (~is_pos) & (p_t >= 0.9)        # already correct with high confidence
    hard = (~is_pos) & (p_t < 0.9)
    return L, dict(easy=L[easy].sum(), hard=L[hard].sum(), pos=L[is_pos].sum(),
                   total=L.sum(), n_easy=int(easy.sum()), n_hard=int(hard.sum()),
                   n_pos=int(is_pos.sum()))

p_t_anch, is_pos_anch = make_anchor_set()
L_ce, ce_b = loss_breakdown(p_t_anch, is_pos_anch, gamma=0.0)              # plain CE: -log(p_t)
L_fl, fl_b = loss_breakdown(p_t_anch, is_pos_anch, gamma=2.0, alpha=0.25)  # the paper's setting

print('anchor set: %s easy negatives | %s hard negatives | %d positives'
      % (format(ce_b['n_easy'], ','), format(ce_b['n_hard'], ','), ce_b['n_pos']))
print('mean foreground prob assigned to an easy negative: %.4f (the model is already right)\n'
      % (1 - p_t_anch[(~is_pos_anch) & (p_t_anch >= 0.9)]).mean())
print('%-26s %14s %14s' % ('', 'CE (gamma=0)', 'FL (g=2,a=.25)'))
print('-' * 58)
for key, lbl in [('easy', 'easy negatives'), ('hard', 'hard negatives'), ('pos', 'positives')]:
    print('%-26s %13.2f %14.2f' % (lbl + ' (sum)', ce_b[key], fl_b[key]))
print('%-26s %13.2f %14.2f' % ('TOTAL', ce_b['total'], fl_b['total']))
print()
for key, lbl in [('easy', 'easy negatives'), ('hard', 'hard negatives'), ('pos', 'positives')]:
    print('%-26s %12.2f%% %13.2f%%' % (lbl + ' (share)', 100*ce_b[key]/ce_b['total'],
                                       100*fl_b[key]/fl_b['total']))
print()
print('easy-negative loss / positive loss:   CE = %6.1fx      FL = %6.1fx'
      % (ce_b['easy']/ce_b['pos'], fl_b['easy']/fl_b['pos']))
print('\n=> Under CE, %d easy negatives that teach NOTHING contribute %.0f%% of the loss and'
      % (ce_b['n_easy'], 100*ce_b['easy']/ce_b['total']))
print('   outweigh the 20 real objects by %.0fx. Focal loss cuts their share to %.2f%% and'
      % (ce_b['easy']/ce_b['pos'], 100*fl_b['easy']/fl_b['total']))
print('   their dominance to %.1fx, WITHOUT discarding a single example.' % (fl_b['easy']/fl_b['pos']))

In [ ]:
fig = plt.figure(figsize=(12.5, 4.4))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.15], height_ratios=[3, 1],
                      hspace=0.12, wspace=0.42)
ax0 = fig.add_subplot(gs[:, 0])     # stacked bar, spans both rows
ax1 = fig.add_subplot(gs[0, 1])     # loss curves
ax2 = fig.add_subplot(gs[1, 1], sharex=ax1)   # anchor density, its own axis (not a second y-scale)

cats = ['easy negatives (%s)' % format(ce_b['n_easy'], ','),
        'hard negatives (%s)' % format(ce_b['n_hard'], ','),
        'positives (%d)' % ce_b['n_pos']]
cols = [C_BLUE, C_YELLOW, C_RED]
for a_i, b_ in enumerate([ce_b, fl_b]):
    bottom = 0.0
    for key, col, cat in zip(['easy', 'hard', 'pos'], cols, cats):
        share = 100 * b_[key] / b_['total']
        ax0.bar([a_i], [share], bottom=bottom, color=col, width=0.5,
                edgecolor='white', linewidth=2, label=cat if a_i == 0 else None)
        if share > 4:
            ax0.text(a_i, bottom + share/2, '%.1f%%' % share, ha='center', va='center',
                     color='white', fontsize=10, weight='bold')
        bottom += share
ax0.set_xticks([0, 1]); ax0.set_xticklabels(['cross entropy', 'focal loss\n(g=2, a=0.25)'])
ax0.set_ylabel('share of total loss (%)'); ax0.set_ylim(0, 100)
ax0.set_title('Where does the loss come from?')
ax0.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=1)
ax0.grid(False)

# WHY the shares move: what a single NEGATIVE anchor is charged...
pp = np.linspace(0.02, 0.9999, 600)
ax1.plot(pp, -np.log(pp), color=C_BLUE, lw=2, label=r'CE:  $-\log(p_t)$')
ax1.plot(pp, -0.75 * (1 - pp)**2 * np.log(pp), color=C_RED, lw=2,
         label=r'FL:  $-\alpha_t(1-p_t)^2\log(p_t)$,  $\alpha_t=0.75$')
ax1.axvline(0.9, color=MUTED, ls='--', lw=1.1)
ax1.set_yscale('log'); ax1.set_ylim(1e-4, 5)
ax1.set_ylabel('loss for one\nnegative anchor')
ax1.set_title('Nearly every anchor sits at $p_t \\approx 1$, where FL charges almost nothing',
              fontsize=10)
ax1.legend(fontsize=8, loc='lower left')
ax1.tick_params(labelbottom=False)

# ...and where the anchors actually are, on its own axis below
ax2.hist(p_t_anch, bins=60, range=(0, 1), color=MUTED, alpha=0.55)
ax2.axvline(0.9, color=MUTED, ls='--', lw=1.1)
ax2.set_yscale('log'); ax2.set_xlim(0, 1)
ax2.set_xlabel('$p_t$'); ax2.set_ylabel('anchors\n(log count)')
ax2.text(0.885, ax2.get_ylim()[1]*0.25, 'easy negatives ->', fontsize=8, color=INK2, ha='right')
plt.show()

The right-hand panel is the mechanism behind the left-hand one: the grey histogram shows that essentially **all** the anchor mass piles up at $p_t \approx 1$. CE (blue) still charges a small but non-zero amount there, and small times 100,000 is what buries the positives. Focal loss (red) has collapsed to nearly zero in exactly that region.

### Slider: $\gamma$, $\alpha$, and the imbalance ratio

Explore the three knobs and watch the accounting recompute live:

- **$\gamma = 0$**: focal loss becomes balanced CE, and the easy-negative share jumps straight back up. This is the control condition.
- **$\gamma$ from 0 to 5**: watch the easy-negative share fall off a cliff. The paper's $\gamma = 2$ sits at the knee of that curve.
- **$\alpha$**: rebalances positives against negatives, but does **not** change the easy/hard split. Note that $\alpha = 0.25$ *down*-weights positives, which seems backwards until you realise $\gamma$ has already done so much work on the negatives that the balance needs correcting the other way.
- **positives** and **hard negative fraction**: make the problem more or less imbalanced and see how CE degrades while FL holds up.

In [ ]:
def show_focal(gamma=2.0, alpha=0.25, n_pos=20, hard_frac=0.01):
    p_t_w, is_pos_w = make_anchor_set(100_000, n_pos, hard_frac, seed=0)
    _, b_fl = loss_breakdown(p_t_w, is_pos_w, gamma, alpha)
    _, b_ce = loss_breakdown(p_t_w, is_pos_w, 0.0)          # plain CE baseline

    fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.7), gridspec_kw={'width_ratios': [1.1, 1, 1]})
    pp = np.linspace(0.001, 1, 500)
    ax[0].plot(pp, -np.log(pp), color=MUTED, ls='--', lw=1.6, label='CE (gamma=0)')
    ax[0].plot(pp, -(1-alpha) * (1-pp)**gamma * np.log(pp), color=C_BLUE,
               label='FL, negatives (a_t=%.2f)' % (1-alpha))
    ax[0].plot(pp, -alpha * (1-pp)**gamma * np.log(pp), color=C_RED,
               label='FL, positives (a_t=%.2f)' % alpha)
    ax[0].set_ylim(0, 3); ax[0].set_xlabel('$p_t$'); ax[0].set_ylabel('loss')
    ax[0].set_title('loss curve (gamma=%.1f, alpha=%.2f)' % (gamma, alpha), fontsize=10)
    ax[0].legend(fontsize=7.5)

    cols_ = [C_BLUE, C_YELLOW, C_RED]
    for a_i, b_ in enumerate([b_ce, b_fl]):
        bottom = 0.0
        for key, col in zip(['easy', 'hard', 'pos'], cols_):
            sh = 100 * b_[key] / b_['total']
            ax[1].bar([a_i], [sh], bottom=bottom, color=col, width=0.5, edgecolor='white', linewidth=2)
            if sh > 5:
                ax[1].text(a_i, bottom + sh/2, '%.0f%%' % sh, ha='center', va='center',
                           color='white', fontsize=9, weight='bold')
            bottom += sh
    ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(['CE', 'FL'])
    ax[1].set_ylim(0, 100); ax[1].set_ylabel('share of loss (%)'); ax[1].grid(False)
    ax[1].set_title('easy=blue, hard=yellow, pos=red', fontsize=9)

    r_ce = b_ce['easy'] / b_ce['pos']; r_fl = b_fl['easy'] / b_fl['pos']
    bars = ax[2].bar(['CE', 'FL'], [r_ce, r_fl], color=[MUTED, C_RED], width=0.5)
    for b_, v_ in zip(bars, [r_ce, r_fl]):
        ax[2].text(b_.get_x()+b_.get_width()/2, v_, ' %.1fx' % v_, ha='center', va='bottom',
                   fontsize=10, color=INK2)
    ax[2].set_yscale('log'); ax[2].set_ylabel('easy-negative loss / positive loss')
    ax[2].set_title('dominance of useless examples\n(lower is better)', fontsize=9)
    plt.tight_layout(); plt.show()

interact(show_focal,
         gamma=widgets.FloatSlider(value=2.0, min=0.0, max=5.0, step=0.25, description='gamma',
                                   continuous_update=False),
         alpha=widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, description='alpha',
                                   continuous_update=False),
         n_pos=widgets.IntSlider(value=20, min=1, max=500, step=1, description='# positives',
                                 continuous_update=False, style={'description_width': '90px'}),
         hard_frac=widgets.FloatSlider(value=0.01, min=0.0, max=0.10, step=0.005,
                                       description='hard neg frac', continuous_update=False,
                                       readout_format='.3f', style={'description_width': '95px'}));

---
# 21. RetinaNet architecture = FPN + focal loss

RetinaNet is deliberately unremarkable as an architecture. That is the point: the paper's claim is that the **loss** was the bottleneck, so the network is assembled from existing parts and only the loss is new.

- **(a) Backbone**: ResNet.
- **(b) Neck**: the feature pyramid net from Section 16.
- **(c) Class subnet**: 4 conv layers of $W \times H \times 256$, then a final conv to $W \times H \times KA$ ($K$ classes, $A$ anchors).
- **(d) Box subnet**: 4 conv layers of $W \times H \times 256$, then a final conv to $W \times H \times 4A$.
- Both subnets are **small FCNs whose weights are shared across all pyramid levels**, though the two subnets do not share with each other. Sharing is what makes "one detector per level" affordable, and it works because FPN already put every level in the same 256-channel semantic space.
- Trained with **focal loss** on the class subnet, which is what makes training over ~100k anchors work at all.

Let us attach these subnets to the FPN we built and count the anchors, to see the ~100k figure appear from real shapes.

In [ ]:
class SubNet(nn.Module):
    "4 x (3x3 conv, 256) then a 3x3 conv to out_ch. Shared across pyramid levels."
    def __init__(self, out_ch, in_ch=256, width=256, n_conv=4):
        super().__init__()
        layers = []
        for _ in range(n_conv):
            layers += [nn.Conv2d(in_ch, width, 3, padding=1), nn.ReLU()]
            in_ch = width
        layers += [nn.Conv2d(width, out_ch, 3, padding=1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

K_CLASSES, A_ANCHORS = 3, 9          # K classes, A anchors per location
torch.manual_seed(SEED)
cls_subnet = SubNet(K_CLASSES * A_ANCHORS).to(device)      # -> W x H x KA
box_subnet = SubNet(4 * A_ANCHORS).to(device)              # -> W x H x 4A

print('RetinaNet head on our FPN (K=%d classes, A=%d anchors per location), input 256x256:\n'
      % (K_CLASSES, A_ANCHORS))
print('%-6s %-20s %-22s %-20s %s' % ('level', 'P_i (from FPN)', 'class subnet -> KA', 'box subnet -> 4A', 'anchors'))
print('-' * 96)
total_anchors = 0
with torch.no_grad():
    for nm, p in [('P3', p3), ('P4', p4), ('P5', p5)]:
        c_out = cls_subnet(p); b_out = box_subnet(p)     # SAME weights reused at every level
        n_a = p.shape[-1] * p.shape[-2] * A_ANCHORS
        total_anchors += n_a
        print('%-6s %-20s %-22s %-20s %s' % (nm, tuple(p.shape), tuple(c_out.shape),
                                             tuple(b_out.shape), format(n_a, ',')))
print('-' * 96)
print('%-6s %-64s %s' % ('TOTAL', '', format(total_anchors, ',')))

n_cls = sum(p.numel() for p in cls_subnet.parameters())
print('\nclass subnet params: %s, and they are SHARED across all 3 levels' % format(n_cls, ','))
print('   (one detector per level, at the parameter cost of one detector in total)')

# extrapolate to the paper's real setting
paper = sum((800 // s)**2 for s in [8, 16, 32, 64, 128]) * A_ANCHORS
print('\nOur toy: %s anchors (256x256 input, levels P3-P5).' % format(total_anchors, ','))
print('RetinaNet paper: 800x800 input, levels P3-P7, A=9  ->  ~%s anchors per image.' % format(paper, ','))
print('THAT is the ~100k number from Section 17, and why focal loss is not optional.')

---
# 22. Detectron

- A framework developed by **Facebook AI Research (FAIR)** to implement state-of-the-art object detection algorithms.
- Highly flexible, providing support across various algorithms and backbone networks.
- See **Detectron** and **Detectron2** for details. Detectron2 ships reference implementations of Faster R-CNN, RetinaNet, Mask R-CNN and more, with pretrained weights, and is the usual starting point for real work rather than reimplementing any of this from scratch.
- (No install here: this notebook is deliberately self-contained. The point of building the pieces by hand was to understand what the framework is doing for you.)

---
# Summary

## Single stage vs two stage

| | Two-stage (Faster R-CNN) | Single-stage (YOLO, SSD, RetinaNet) |
|---|---|---|
| Pipeline | propose regions, then classify | one dense forward pass |
| Candidates scored | ~1k to 2k proposals | ~10k (YOLO/SSD) to ~100k (RetinaNet) anchors |
| Foreground:background | pre-filtered by the proposal stage, roughly manageable | ~1:1000 or worse, nothing filters it |
| Speed | slower | real time |
| Key weakness | speed | class imbalance, which cost accuracy until focal loss |

## The three detectors of this lecture

| | **YOLO (v1)** | **SSD** | **RetinaNet** |
|---|---|---|---|
| Paper | Redmon et al, CVPR 2016 | Liu et al, ECCV 2016 | Lin et al, ICCV 2017 |
| Prediction sites | $S \times S$ grid, one scale | multi-scale feature maps | FPN levels P3 to P7 |
| Output per cell | $B \cdot 5 + C$ (classes **shared** across boxes) | $(c + 4)k$ per location (classes **per box**) | $KA$ + $4A$, from shared subnets |
| Priors | none, boxes regressed directly | default boxes, $k$ aspect ratios | 9 anchors per location |
| Box encoding | $(x,y)$ per cell, $(w,h)$ per image, $\sqrt{\cdot}$ on size | offsets from the default box, log space | offsets from the anchor, log space |
| Class loss | sum-squared error on softmax | softmax CE + **hard negative mining** (3:1) | **focal loss**, $\gamma=2$, $\alpha=0.25$ |
| Box loss | sum-squared error, $\lambda_{\text{coord}}=5$ | smooth L1 | smooth L1 |
| Imbalance fix | $\lambda_{\text{noobj}} = 0.5$ | discard easy negatives | down-weight them smoothly |
| Small objects | poor (single coarse scale) | better (multi-scale) | good (FPN) |

## The through-line

Read the "imbalance fix" row across: $\lambda_{\text{noobj}} = 0.5$, then 3:1 mining, then focal loss. **The same problem, attacked three times with progressively less crude tools.** The lecture's arc is not really "three detectors", it is one unresolved consequence of dropping the proposal stage, chased until someone named it properly.

Three ideas worth carrying forward:
- **Detection is a dense prediction problem.** The output tensor is just a feature map with a particular channel layout. Once you see the $S \times S \times (B \cdot 5 + C)$ tensor as a conv output, single-stage detection stops being mysterious.
- **Parameterisation is design.** Sigmoid-bounded centres, log-space sizes, anchor priors and $\sqrt{w}$ are all attempts to make box regression an *easy* function to learn. The network is the same; the coordinates you ask it for are not.
- **The loss is where the modelling assumptions live.** Softmax versus per-class logistic decides whether labels can co-occur. $\lambda_{\text{noobj}}$, mining, and focal loss decide which examples matter. Changing a loss changed the state of the art without touching the architecture.

---
# Exercises

**1. YOLO9000 and YOLOv3.** YOLO9000 and YOLOv3 were follow-ups of YOLOv2. What was different in these extensions? Find out.
*(Hint: see the "YOLO Family: All you want to know" reading. Consider: how does YOLO9000 train on classification and detection data jointly, and what is the WordTree hierarchical softmax for? Section 9 above covers the v3 side of the answer, so focus on v2 to 9000.)*

**2. IoU by hand.** Given two bounding boxes in an image: an upper-left box which is $2 \times 2$, and a lower-right box which is $2 \times 3$, with an overlapping region of $1 \times 1$. What is the IoU between the two boxes?

<details>
<summary>Solution</summary>

$$
\text{IoU} = \frac{\text{Intersection}}{\text{Union}} = \frac{1}{(2 \times 2) + (2 \times 3) - 1} = \frac{1}{4 + 6 - 1} = \frac{1}{9} \approx 0.111
$$

The trap is forgetting to subtract the intersection from the union: $4 + 6 = 10$ double-counts the shared cell. We verified this exact case numerically against our `iou()` function in Section 6.1.
</details>

**3. Output volume size.** Consider using a YOLO object detector on a $19 \times 19$ grid, on a detection problem with 20 classes, and with 5 anchor boxes. During training, for each image, you construct an output volume $y$ as the target value for the neural network, corresponding to the last layer of the network ($y$ may include background). What is the dimension of this output volume?

<details>
<summary>Solution</summary>

With anchor boxes (YOLO v2 style) each anchor gets its **own** 5 box values *and* its own class scores, unlike v1 where the $C$ class probabilities are shared per cell:

$$
19 \times 19 \times \big[\, 5 \times (5 + 20) \,\big] = 19 \times 19 \times 125 = 45{,}125
$$

The inner $5 + 20$ is $(p_c, b_x, b_y, b_w, b_h)$ plus the 20 class scores. Note how this differs from the v1 layout we implemented in Section 4, which would give $19 \times 19 \times (5 \cdot 5 + 20) = 19 \times 19 \times 45$: that is exactly the "one class per cell" restriction anchors removed. Use the $S$/$B$/$C$ slider in Section 3 to compare the two layouts.
</details>

**4. (Optional, hands-on.)** Extend the mini-YOLO you trained in Section 6:
- Swap the softmax + SSE class loss for **independent per-class logistic classifiers with BCE**, as YOLO v3 does. Does anything change on this dataset, and why or why not?
- Set $\lambda_{\text{noobj}} = 1.0$ and retrain. Watch the `noobj` term in the loss curve and the resulting recall.
- Remove the rejection-sampling constraint that forbids **two object centres in one cell**, then look at what the detector does with those images. You have now reproduced YOLO v1's most famous limitation on purpose.

---

### Readings

- Object Detection for Dummies, Part 4
- YOLO Family: All you want to know
- Understanding SSD, Understanding FPN, Understanding RetinaNet

### References

1. Wei Liu et al. "SSD: Single shot multibox detector". ECCV 2016, pp. 21 to 37.
2. Joseph Redmon et al. "You only look once: Unified, real-time object detection". CVPR 2016, pp. 779 to 788.
3. Tsung-Yi Lin et al. "Feature pyramid networks for object detection". CVPR 2017, pp. 2117 to 2125.
4. Tsung-Yi Lin et al. "Focal loss for dense object detection". ICCV 2017, pp. 2980 to 2988.
5. Joseph Redmon and Ali Farhadi. "YOLO9000: better, faster, stronger". CVPR 2017, pp. 7263 to 7271.
6. Terven & Cordova-Esparza. "A Comprehensive Review of YOLO Architectures in Computer Vision: From YOLOv1 to YOLOv8 and YOLO-NAS". arXiv 2024.